# Orquestração e Evolução para Sistema Multiagente
## Banco Aurora — do agente único ReAct ao sistema multiagente coordenado

**Disciplina:** Agentes e Workflows com LLM — Entrega II.

**Entrega anterior (I):** um agente único ReAct que respondia perguntas de negócio
consultando um data warehouse SQLite em memória, com quatro ferramentas, guardrails
de entrada/SQL, fallback de modelos e testes com `MockLLM`.

**Esta entrega:** evolui a solução em duas etapas explícitas:

1. **Orquestração (C2):** o agente único é reorganizado dentro de um `StateGraph`
   do **LangGraph**, ganhando estado explícito, checkpoint e trilha de auditoria,
   sem mudar sua responsabilidade.
2. **Decomposição (C3):** a responsabilidade é dividida em um **Coordenador** e
   três especialistas com fronteiras justificadas: **Domínio**, **Analista SQL** e
   **Auditor de Resultados**.

A avaliação compara três configurações nos mesmos cenários:

| Configuração | O que é | O que isola |
|---|---|---|
| **C1** | Loop ReAct manual da Entrega I | Ponto de partida |
| **C2** | Mesmo agente único dentro do LangGraph | Ganho da orquestração |
| **C3** | Coordenador + Domínio + Analista SQL + Auditor | Ganho da decomposição |

> **Único requisito externo:** a chave da API do LLM. Todos os testes e a avaliação
> offline rodam sem chave, com `MockLLM` determinístico.

---

## 1. Por que multiagente neste problema?

O agente único da Entrega I concentra quatro trabalhos diferentes no mesmo prompt
e no mesmo contexto:

- interpretar a pergunta e decidir o escopo;
- resolver a semântica de negócio (o que é churn, ticket médio, transação válida);
- escrever e corrigir SQL;
- verificar se o resultado realmente responde à pergunta.

Esses objetivos conflitam. O mesmo agente que escreveu o SQL tende a aprová-lo;
definições de negócio competem por espaço com o schema; e não há como aplicar
permissões ou políticas diferentes por papel. Em BI bancário, o custo de um número
errado é alto, e a auditabilidade é parte do requisito.

A decomposição proposta separa **conhecimento**, **ação** e **verificação**:

| Agente | Responsabilidade | Ferramentas | Por que é independente |
|---|---|---|---|
| **Coordenador** | Interpretar, planejar, delegar, controlar orçamento e decidir entre responder, revisar ou escalar | Nenhuma ferramenta de dados | Ponto único de decisão e accountability; concentra política, orçamento e resposta final; impede que especialistas ampliem escopo |
| **Especialista de Domínio** | Resolver termos, métricas, tabelas/colunas e ambiguidades; produzir um *brief* estruturado | `inspecionar_schema`, `buscar_documentacao` | Conhecimento semântico é um recurso distinto do raciocínio SQL; evita que o gerador de SQL invente a definição de uma métrica e permite versionar/cachear a camada semântica |
| **Analista SQL** | Formular, executar e corrigir SQL somente leitura; devolver SQL, resultado, hipóteses e erros | `executar_sql` | É a ação de maior risco e tem ciclo próprio de tentativa/erro; permite permissões e limites distintos e testes específicos de SQL |
| **Auditor de Resultados** | Verificar aderência pergunta–SQL–resultado, filtros, período, agregação e plausibilidade; aprovar ou pedir revisão | Checagens determinísticas + LLM | Separação *maker-checker*: quem escreveu o SQL não certifica o próprio resultado; reduz erro semântico silencioso e alucinação |

**Não há “um agente por ferramenta”.** Guardrails, política de dados e auditoria
são componentes determinísticos. O Analista SQL usa apenas uma ferramenta, mas sua
responsabilidade é mais ampla: interpretar o brief, formular a consulta, executar,
diagnosticar erro e corrigir.

### Por que LangGraph?

| Alternativa | O que oferece | Por que não foi escolhida |
|---|---|---|
| **LangGraph** | Grafo de estado tipado, conditional edges, checkpointer, `interrupt`, composição hierárquica | **Escolhida:** contratos, roteamento e auditoria ficam explícitos no código, que é o que a rubrica pede |
| CrewAI | Papéis/tarefas e delegação prontos | A coordenação fica mais escondida em conversas; dificulta demonstrar contratos e trilha de decisão |
| AutoGen | Conversas entre agentes | Modelo conversacional dificulta impor payloads estruturados e política determinística |
| Loop manual | Controle total | Não demonstra um framework de orquestração nem checkpoint/HITL |

Também avaliamos usar o pacote `langgraph-supervisor` pronto. Preferimos montar o
`StateGraph` explicitamente porque o problema exige contratos validáveis, política
antes do SQL e um loop de revisão com feedback — tudo mais claro quando os nós e as
arestas estão no notebook.

### Trade-off assumido

Multiagente custa mais latência e mais chamadas de LLM. Para perguntas triviais, o
agente único é mais barato. A aposta arquitetural é que, em perguntas dependentes de
definição, com risco de erro silencioso e necessidade de auditoria, a separação se
paga. A Seção de avaliação mede esse trade-off em vez de apenas afirmá-lo.

---

## 2. Arquitetura de coordenação

```mermaid
flowchart TD
    U[Pergunta] --> V[Validação de entrada]
    V --> C[Coordenador: plano]
    C -->|definição ou consulta| D[Especialista de Domínio]
    D -->|brief + lacunas| P[Policy Gate]
    P --> A[Analista SQL]
    A -->|SQL + resultado + hipóteses| R[Auditor]
    R -->|parecer| C2[Coordenador: decisão]
    C2 -->|aprovado| F[Resposta final + proveniência]
    C2 -->|revisar| A
    C2 -->|escalar| E[Ticket humano]
    C2 -->|esclarecer| F
    D -->|definição pura| F
    D -->|lacuna crítica| Q[Pedido de esclarecimento]
```

### Contratos de interação

Toda passagem entre agentes usa um **envelope** com identidade, correlação,
remetente, destinatário, tipo, payload e proveniência. O payload é validado por um
schema Pydantic. Texto livre não é a fonte de verdade: é apenas a explicação.

| Fronteira | Contrato | Campos críticos |
|---|---|---|
| Coordenador → Domínio | `PlanoTarefa` | tipo, objetivo, termos, período, risco |
| Domínio → Analista/Auditor | `BriefDominio` | definições, tabelas, colunas, regras, lacunas, confiança |
| Analista → Auditor | `ArtefatoConsulta` | SQL, resultado, hipóteses, erro, tentativas |
| Auditor → Coordenador | `ParecerVerificacao` | aprovado, motivos, problemas, correção sugerida |
| Coordenador → usuário | `RespostaFinal` | resposta, evidências, SQL, definições, ressalvas |

### Mecanismos de coordenação

- **Supervisor:** o Coordenador decide o próximo passo por *conditional edges*.
- **Loop de revisão:** Auditor reprova → Coordenador envia feedback específico →
  Analista reexecuta, com teto de revisões.
- **Preservação de informação:** SQL, linhas, definições, hipóteses, erros e feedback
  permanecem no estado e nos envelopes; a resposta final cita a proveniência.
- **Orçamento:** máximo de tentativas de SQL, revisões e passos por especialista.
- **Auditoria:** log append-only com `correlation_id`, transições e decisões.
- **Checkpoint:** `InMemorySaver` + `thread_id`, permitindo inspecionar e reproduzir.
- **Política:** gate antes da ação SQL, com tabelas permitidas e mascaramento de PII.

### Preparação para a governança (Entrega III)

Esta entrega **não** implementa a governança completa. Ela deixa os pontos de
acoplamento prontos: classificação de risco no plano, `policy_gate` antes do SQL,
trilha de auditoria, checkpoint/replay e um ponto claro para `interrupt` de aprovação
humana. A Seção final detalha o que entra na próxima etapa.

## 3. Setup

A célula abaixo garante o LangGraph no Colab. Se as bibliotecas já estiverem
instaladas, ela não faz nada. Depois, a chave é lida do Colab Secrets ou de uma
variável de ambiente; se não houver chave, o notebook segue em modo offline.

São aceitos dois provedores, detectados pelo prefixo da chave:

- **OpenRouter** (`sk-or-...`) — modelos gratuitos com descoberta dinâmica;
- **OpenCode Zen** (`oc_sk_...`) — modelos como `deepseek-v4.1-flash` e
  `claude-haiku-5-5`.

Configure o secret com o nome `OPENROUTER_API_KEY`, `OPENCODE_API_KEY` ou
`LLM_API_KEY`, ou exporte a variável no ambiente.

In [1]:
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

def _versao(pacote):
    try:
        return _md.version(pacote)
    except Exception:
        return None

_versao_langgraph = _versao("langgraph")
_versao_openai = _versao("openai")
_precisa_instalar = (
    _versao_langgraph is None
    or int(_versao_langgraph.split(".")[0]) < 1
    or _versao_openai is None
)
if _precisa_instalar:
    print("Instalando LangGraph e openai...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "langgraph==1.2.20", "openai",
    ])

import langgraph
print("langgraph:", _versao("langgraph"))
print("openai:", _versao("openai"))
print("pydantic:", _versao("pydantic"))

langgraph: 1.2.20
openai: 3.46.0
pydantic: 2.14.3


In [2]:
import os
import sys

_chave = ""
for _nome in ("OPENROUTER_API_KEY", "OPENCODE_API_KEY", "LLM_API_KEY"):
    _chave = os.environ.get(_nome, "").strip()
    if _chave:
        break

if not _chave:
    try:
        from google.colab import userdata  # type: ignore
        for _nome in ("OPENROUTER_API_KEY", "OPENCODE_API_KEY", "LLM_API_KEY"):
            try:
                _chave = (userdata.get(_nome) or "").strip()
            except Exception:
                pass
            if _chave:
                break
    except Exception:
        pass

# Em execução não interativa (nbconvert/CI) não bloqueia esperando digitação.
if not _chave and sys.stdin.isatty():
    try:
        import getpass
        _chave = getpass.getpass(
            "OPENROUTER_API_KEY ou OPENCODE_API_KEY "
            "(ENTER para rodar apenas os testes offline): "
        ).strip()
    except Exception:
        _chave = ""

os.environ["LLM_API_KEY"] = _chave
_provedor = (
    "OpenCode Zen" if _chave.startswith("oc_sk_")
    else "OpenRouter" if _chave.startswith("sk-or-")
    else "nao detectado" if _chave else "offline"
)
print("Chave configurada:", bool(_chave))
print("Provedor detectado:", _provedor)
print("Modo:", "demo real habilitada" if _chave else "offline (MockLLM)")

Chave configurada: True
Provedor detectado: OpenCode Zen
Modo: demo real habilitada


## 4. Base reaproveitada da Entrega I

Mantemos o data warehouse determinístico, o glossário, as ferramentas e os
guardrails. A única extensão de política é o mascaramento de PII e a avaliação de
tabelas permitidas antes da execução SQL — um gancho para a governança final.

In [3]:
import json
import operator
import random
import re
import sqlite3
import time
import unicodedata
import uuid
from datetime import datetime, timezone
from typing import Annotated, Any, Literal, TypedDict

from pydantic import BaseModel, ConfigDict, Field, ValidationError

MAX_ENTRADA = 500
MAX_ROWS = 200
MAX_STEPS = 6
MAX_SQL_RETRIES = 2
TOOL_TIMEOUT = 10
MAX_REVISOES = 2
MAX_TENTATIVAS_SQL = 2
MAX_PASSOS_ESPECIALISTA = 4

_PADROES_INJECAO = [
    "ignore previous instructions",
    "ignore prior instructions",
    "ignore all previous",
    "ignore as instrucoes",
    "esqueca tudo",
    "esqueca as regras",
    "agora voce e um",
    "system prompt:",
    "jailbreak",
    "dan mode",
    "<script",
]

_PADROES_DESTRUTIVOS = [
    "apague", "apagar", "delete", "deletar", "remova", "remover",
    "drop", "truncate", "insira", "inserir", "update", "altere",
    "alterar", "atualize", "atualizar",
]

def _normalizar(texto: str) -> str:
    return unicodedata.normalize("NFKC", texto or "").lower()

def validar_entrada(texto):
    if not texto or not str(texto).strip():
        raise ValueError("A pergunta nao pode estar vazia.")
    t = str(texto).strip()
    if len(t) > MAX_ENTRADA:
        raise ValueError(f"A pergunta excede o limite de {MAX_ENTRADA} caracteres.")
    normalizado = _normalizar(t)
    for padrao in _PADROES_INJECAO:
        if padrao in normalizado:
            raise ValueError("Entrada bloqueada por politica anti-prompt-injection.")
    return t

def detectar_intencao_destrutiva(texto: str) -> bool:
    normalizado = _normalizar(texto)
    return any(p in normalizado for p in _PADROES_DESTRUTIVOS)

class SQLInseguro(ValueError):
    pass

_COMANDOS_PROIBIDOS = {
    "insert", "update", "delete", "drop", "alter", "create", "replace",
    "truncate", "attach", "detach", "pragma", "vacuum", "grant", "revoke",
    "exec", "execute",
}

def validar_sql(sql):
    s = (sql or "").strip()
    while s.endswith(";"):
        s = s[:-1].strip()
    if not s:
        raise SQLInseguro("SQL vazio.")
    low = s.lower()
    if not low.startswith("select"):
        raise SQLInseguro("Apenas consultas SELECT sao permitidas.")
    if ";" in s:
        raise SQLInseguro("Apenas uma instrucao por consulta.")
    tokens = set(re.findall("[a-z_]+", low))
    proibidos = tokens.intersection(_COMANDOS_PROIBIDOS)
    if proibidos:
        raise SQLInseguro(
            "Comando nao permitido nesta base somente leitura: "
            + ", ".join(sorted(proibidos))
        )
    if " limit " not in low:
        s = s + " LIMIT " + str(MAX_ROWS)
    return s

POLITICAS = {
    "tabelas_permitidas": {
        "clientes", "transacoes", "faturas", "solicitacoes_limite",
    },
    "colunas_pii": {
        "clientes": ["nome"],
    },
    "max_linhas": MAX_ROWS,
}

def criar_dw():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript(
        "CREATE TABLE clientes (id INTEGER PRIMARY KEY, nome TEXT, cidade TEXT, uf TEXT, renda_mensal REAL, score INTEGER, data_cadastro TEXT);"
        "CREATE TABLE transacoes (id INTEGER PRIMARY KEY, cliente_id INTEGER, data TEXT, valor REAL, tipo TEXT, moeda TEXT);"
        "CREATE TABLE faturas (id INTEGER PRIMARY KEY, cliente_id INTEGER, mes_referencia TEXT, valor_total REAL, paga INTEGER);"
        "CREATE TABLE solicitacoes_limite (id INTEGER PRIMARY KEY, cliente_id INTEGER, data_solicitacao TEXT, status TEXT, limite_novo REAL);"
    )
    return conn

def semear(conn):
    rng = random.Random(42)
    cidades = [
        ("Sao Paulo", "SP"), ("Rio de Janeiro", "RJ"), ("Recife", "PE"),
        ("Curitiba", "PR"), ("Salvador", "BA"),
    ]
    clientes = []
    for cid in range(1, 21):
        cidade, uf = cidades[rng.randrange(len(cidades))]
        renda = rng.choice([1500.0, 2000.0, 3000.0, 4500.0, 6000.0, 9000.0])
        score = rng.choice([450, 550, 620, 700, 780])
        dia = str(rng.randint(1, 28)).zfill(2)
        clientes.append((
            cid, "Cliente " + str(cid).zfill(2), cidade, uf, renda, score,
            "2026-01-" + dia,
        ))
    conn.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?,?)", clientes)

    tipos = ["credito", "debito", "pix", "saque"]
    transacoes = []
    tid = 1
    for cid in range(1, 21):
        for _ in range(rng.randint(3, 9)):
            mes = str(rng.randint(1, 6)).zfill(2)
            dia = str(rng.randint(1, 28)).zfill(2)
            base = rng.choice([10.0, 25.5, 120.0, 300.0, 1200.0, 2500.0])
            valor = round(base * rng.uniform(0.8, 1.2), 2)
            transacoes.append((
                tid, cid, "2026-" + mes + "-" + dia, valor,
                rng.choice(tipos), "BRL",
            ))
            tid += 1
    conn.executemany("INSERT INTO transacoes VALUES (?,?,?,?,?,?)", transacoes)

    faturas = []
    fid = 1
    for cid in range(1, 21):
        for mes in (4, 5, 6):
            faturas.append((
                fid, cid, "2026-" + str(mes).zfill(2),
                round(rng.uniform(80, 3000), 2), rng.choice([0, 1]),
            ))
            fid += 1
    conn.executemany("INSERT INTO faturas VALUES (?,?,?,?,?)", faturas)

    solicitacoes = []
    for sid in range(1, 11):
        cid = rng.randint(1, 20)
        mes = str(rng.randint(1, 6)).zfill(2)
        dia = str(rng.randint(1, 28)).zfill(2)
        solicitacoes.append((
            sid, cid, "2026-" + mes + "-" + dia,
            rng.choice(["aprovada", "recusada", "em_analise"]),
            rng.choice([1000.0, 2500.0, 5000.0]),
        ))
    conn.executemany("INSERT INTO solicitacoes_limite VALUES (?,?,?,?,?)", solicitacoes)
    conn.commit()

DW = criar_dw()
semear(DW)

GLOSSARIO = {
    "churn": "Cliente sem nenhuma transacao nos ultimos 30 dias em relacao a data de referencia do DW.",
    "transacao valida": "Transacao com valor maior que zero e moeda BRL.",
    "ticket medio": "Media de valor por transacao valida.",
    "limite": "Solicitacoes de aumento de limite ficam na tabela solicitacoes_limite.",
    "base": "DW Aurora com as tabelas clientes, transacoes, faturas e solicitacoes_limite.",
}

def resumo_catalogo() -> str:
    linhas = []
    tabelas = [
        r[0] for r in DW.execute(
            "SELECT name FROM sqlite_master WHERE type = ? ORDER BY name", ("table",)
        ).fetchall()
    ]
    for tabela in tabelas:
        colunas = [
            r["name"] + ":" + r["type"]
            for r in DW.execute("PRAGMA table_info(" + tabela + ")").fetchall()
        ]
        linhas.append("- " + tabela + " (" + ", ".join(colunas) + ")")
    return "\n".join(linhas)

CATALOGO = resumo_catalogo()

In [4]:
def inspecionar_schema(tabela=None):
    try:
        tabelas = [
            r[0] for r in DW.execute(
                "SELECT name FROM sqlite_master WHERE type = ? ORDER BY name",
                ("table",),
            ).fetchall()
        ]
        if tabela is None:
            return {"ok": True, "data": {"tabelas": tabelas}}
        if tabela not in tabelas:
            return {"ok": False, "error": "Tabela inexistente: " + str(tabela)}
        colunas = [
            {"coluna": r["name"], "tipo": r["type"]}
            for r in DW.execute("PRAGMA table_info(" + tabela + ")").fetchall()
        ]
        return {"ok": True, "data": {"tabela": tabela, "colunas": colunas}}
    except Exception as e:
        return {"ok": False, "error": "Falha ao inspecionar schema: " + str(e)}

def buscar_documentacao(termo):
    t = _normalizar(termo).strip()
    achados = {}
    for chave, valor in GLOSSARIO.items():
        if t and (t in chave or chave in t):
            achados[chave] = valor
    if not achados:
        return {"ok": False, "error": "Nenhuma definicao encontrada para: " + str(termo)}
    return {"ok": True, "data": achados}

def avaliar_politica_sql(sql: str) -> dict:
    s = _normalizar(sql)
    tabelas = set(re.findall(r"\b(?:from|join)\s+([a-z_][a-z0-9_]*)", s))
    nao_permitidas = sorted(tabelas - POLITICAS["tabelas_permitidas"])
    if nao_permitidas:
        return {
            "permitido": False,
            "motivos": ["Tabela(s) fora da politica: " + ", ".join(nao_permitidas)],
            "mascarar_pii": False,
        }
    pii = False
    for tabela, colunas in POLITICAS["colunas_pii"].items():
        if tabela in tabelas:
            for coluna in colunas:
                if re.search(r"\b" + re.escape(coluna) + r"\b", s):
                    pii = True
                    break
    return {
        "permitido": True,
        "motivos": [],
        "mascarar_pii": pii,
    }

def mascarar_pii(resultado: dict) -> dict:
    if not resultado.get("ok"):
        return resultado
    data = resultado.get("data", {})
    colunas_pii = {
        coluna for colunas in POLITICAS["colunas_pii"].values() for coluna in colunas
    }
    mascarou = False
    novas_linhas = []
    for linha in data.get("rows", []):
        nova = dict(linha)
        for coluna in list(nova):
            if _normalizar(coluna) in colunas_pii:
                nova[coluna] = "***"
                mascarou = True
        novas_linhas.append(nova)
    data["rows"] = novas_linhas
    data["pii_mascarada"] = mascarou
    resultado["data"] = data
    return resultado

def executar_sql(sql):
    try:
        seguro = validar_sql(sql)
    except SQLInseguro as e:
        return {"ok": False, "error": str(e)}

    politica = avaliar_politica_sql(seguro)
    if not politica["permitido"]:
        return {
            "ok": False,
            "error": "Politica de dados: " + "; ".join(politica["motivos"]),
            "politica": politica,
        }

    prazo = time.monotonic() + TOOL_TIMEOUT

    def _handler():
        return 1 if time.monotonic() > prazo else 0

    try:
        DW.set_progress_handler(_handler, 1000)
        cur = DW.execute(seguro)
        colunas = [d[0] for d in (cur.description or [])]
        linhas = [
            dict(zip(colunas, tuple(r))) for r in cur.fetchmany(MAX_ROWS)
        ]
        resultado = {
            "ok": True,
            "data": {
                "columns": colunas,
                "rows": linhas,
                "n": len(linhas),
                "sql": seguro,
                "politica": politica,
            },
        }
        if politica.get("mascarar_pii"):
            resultado = mascarar_pii(resultado)
        return resultado
    except sqlite3.Error as e:
        return {"ok": False, "error": "Erro de SQL: " + str(e)}
    finally:
        DW.set_progress_handler(None, 1000)

_CONTADOR_TICKET = {"n": 0}

def escalar_analista(motivo):
    _CONTADOR_TICKET["n"] += 1
    return {
        "ok": True,
        "data": {
            "ticket": "ANL-" + str(_CONTADOR_TICKET["n"]).zfill(3),
            "motivo": motivo,
        },
    }

FERRAMENTAS = {
    "inspecionar_schema": {
        "fn": inspecionar_schema,
        "descricao": "Lista as tabelas do DW ou as colunas de uma tabela.",
        "parametros": {"tabela": "nome da tabela, opcional"},
    },
    "buscar_documentacao": {
        "fn": buscar_documentacao,
        "descricao": "Busca definicoes de metricas e regras de negocio.",
        "parametros": {"termo": "metrica ou palavra-chave"},
    },
    "executar_sql": {
        "fn": executar_sql,
        "descricao": "Executa um SELECT somente leitura no DW.",
        "parametros": {"sql": "consulta SELECT"},
    },
    "escalar_analista": {
        "fn": escalar_analista,
        "descricao": "Encaminha o caso para um analista humano.",
        "parametros": {"motivo": "motivo do encaminhamento"},
    },
}

def executar_ferramenta(nome, argumentos):
    if nome not in FERRAMENTAS:
        return {"ok": False, "error": "Ferramenta desconhecida: " + str(nome)}
    if argumentos is None:
        argumentos = {}
    if not isinstance(argumentos, dict):
        return {"ok": False, "error": "Os argumentos devem ser um objeto JSON."}
    try:
        return FERRAMENTAS[nome]["fn"](**argumentos)
    except TypeError as e:
        return {"ok": False, "error": "Argumentos invalidos para " + nome + ": " + str(e)}
    except Exception as e:
        return {"ok": False, "error": "Falha na ferramenta " + nome + ": " + str(e)}

def descricao_ferramentas():
    linhas = []
    for nome, spec in FERRAMENTAS.items():
        linhas.append(
            "- " + nome + ": " + spec["descricao"]
            + " | parametros: " + str(spec["parametros"])
        )
    return "\n".join(linhas)

print("DW em memoria criado.")
for nome in ("clientes", "transacoes", "faturas", "solicitacoes_limite"):
    n = DW.execute("SELECT COUNT(*) FROM " + nome).fetchone()[0]
    print("  ", nome, "->", n, "linhas")
print("Ferramentas:", list(FERRAMENTAS.keys()))
print("Catalogo:\n" + CATALOGO)

DW em memoria criado.
   clientes -> 20 linhas
   transacoes -> 136 linhas
   faturas -> 60 linhas
   solicitacoes_limite -> 10 linhas
Ferramentas: ['inspecionar_schema', 'buscar_documentacao', 'executar_sql', 'escalar_analista']
Catalogo:
- clientes (id:INTEGER, nome:TEXT, cidade:TEXT, uf:TEXT, renda_mensal:REAL, score:INTEGER, data_cadastro:TEXT)
- faturas (id:INTEGER, cliente_id:INTEGER, mes_referencia:TEXT, valor_total:REAL, paga:INTEGER)
- solicitacoes_limite (id:INTEGER, cliente_id:INTEGER, data_solicitacao:TEXT, status:TEXT, limite_novo:REAL)
- transacoes (id:INTEGER, cliente_id:INTEGER, data:TEXT, valor:REAL, tipo:TEXT, moeda:TEXT)


## 5. Contratos, envelopes e estado do grafo

Os contratos são a parte central da coordenação. Cada agente recebe um payload
validado e devolve outro payload validado. O envelope acrescenta identidade e
proveniência, permitindo reconstruir a cadeia de decisões.

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

class EnvelopeMensagem(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str
    correlation_id: str
    de: str
    para: str
    tipo: str
    timestamp: str
    payload: dict
    proveniencia: list[str] = Field(default_factory=list)


class PlanoTarefa(BaseModel):
    model_config = ConfigDict(extra="forbid")

    tipo: Literal["consulta", "definicao", "fora_escopo", "invalido"]
    objetivo: str
    termos: list[str] = Field(default_factory=list)
    periodo: str | None = None
    entidades: list[str] = Field(default_factory=list)
    precisa_dominio: bool = True
    risco: Literal["baixo", "medio", "alto"] = "baixo"
    justificativa: str = ""


class BriefDominio(BaseModel):
    model_config = ConfigDict(extra="forbid")

    definicoes: dict[str, str] = Field(default_factory=dict)
    tabelas: list[str] = Field(default_factory=list)
    colunas: dict[str, list[str]] = Field(default_factory=dict)
    regras: list[str] = Field(default_factory=list)
    lacunas: list[str] = Field(default_factory=list)
    confianca: float = 0.0


class ArtefatoParcial(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sql: str = ""
    explicacao: str = ""
    hipoteses: list[str] = Field(default_factory=list)


class ArtefatoConsulta(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sql: str
    resultado: dict
    explicacao: str
    hipoteses: list[str] = Field(default_factory=list)
    erro: str | None = None
    tentativas: int = 1
    tool_calls: list[str] = Field(default_factory=list)


class ParecerVerificacao(BaseModel):
    model_config = ConfigDict(extra="forbid")

    aprovado: bool
    motivos: list[str] = Field(default_factory=list)
    problemas: list[str] = Field(default_factory=list)
    correcao_sugerida: str | None = None
    confianca: float = 0.0


class DecisaoCoordenador(BaseModel):
    model_config = ConfigDict(extra="forbid")

    acao: Literal["finalizar", "revisar", "escalar", "esclarecer"]
    resposta: str | None = None
    instrucao: str | None = None
    motivo: str = ""


class EstadoAurora(TypedDict, total=False):
    pergunta: str
    correlation_id: str
    thread_id: str
    plano: dict
    brief: dict
    artefato: dict
    parecer: dict
    decisao: dict
    resposta: str
    status: str
    erro: str
    trace: list
    revisoes: int
    tentativas_sql: int
    mensagens: Annotated[list[dict], operator.add]
    auditoria: Annotated[list[dict], operator.add]
    tool_calls: Annotated[list[str], operator.add]


def novo_id(prefixo: str) -> str:
    return prefixo + "-" + uuid.uuid4().hex[:12]


def agora_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def novo_envelope(de, para, tipo, payload, correlation_id, proveniencia=None):
    return EnvelopeMensagem(
        id=novo_id("msg"),
        correlation_id=correlation_id,
        de=de,
        para=para,
        tipo=tipo,
        timestamp=agora_iso(),
        payload=payload,
        proveniencia=list(proveniencia or []),
    ).model_dump()


def resumir_payload(payload: dict) -> dict:
    resumo = {"chaves": sorted(payload.keys())}
    if "sql" in payload:
        resumo["sql"] = str(payload["sql"])[:220]
    if "resultado" in payload and isinstance(payload["resultado"], dict):
        resumo["n_linhas"] = payload["resultado"].get("n", 0)
        resumo["colunas"] = payload["resultado"].get("columns", [])
    if "definicoes" in payload and isinstance(payload["definicoes"], dict):
        resumo["definicoes"] = list(payload["definicoes"].keys())
    if "lacunas" in payload:
        resumo["lacunas"] = payload.get("lacunas", [])
    if "aprovado" in payload:
        resumo["aprovado"] = payload["aprovado"]
    if "acao" in payload:
        resumo["acao"] = payload["acao"]
    return resumo


def entrada_auditoria(envelope: dict) -> dict:
    return {
        "evento": "handoff",
        "envelope_id": envelope["id"],
        "correlation_id": envelope["correlation_id"],
        "de": envelope["de"],
        "para": envelope["para"],
        "tipo": envelope["tipo"],
        "timestamp": envelope["timestamp"],
        "resumo": resumir_payload(envelope["payload"]),
    }


def empacotar(state: dict, de, para, tipo, payload, proveniencia=None) -> dict:
    env = novo_envelope(
        de, para, tipo, payload, state["correlation_id"], proveniencia
    )
    return {"mensagens": [env], "auditoria": [entrada_auditoria(env)]}


print("Contratos carregados:", [
    EnvelopeMensagem.__name__, PlanoTarefa.__name__, BriefDominio.__name__,
    ArtefatoConsulta.__name__, ParecerVerificacao.__name__,
    DecisaoCoordenador.__name__,
])

Contratos carregados: ['EnvelopeMensagem', 'PlanoTarefa', 'BriefDominio', 'ArtefatoConsulta', 'ParecerVerificacao', 'DecisaoCoordenador']


## 6. Camada LLM

Mantemos dois clientes compatíveis com APIs OpenAI: **OpenRouter** (chave
`sk-or-...`) e **OpenCode Zen** (chave `oc_sk_...`). A escolha é automática pelo
prefixo da chave. Para os testes offline, um `MockLLM` roteiriza respostas por
agente, sem chave e sem chamadas de rede.

A função `chamar_json` valida o contrato e tenta uma reparação. A função
`loop_react` implementa o ciclo decisão → ação → observação dentro de cada
especialista, com um schema de saída obrigatório.

In [6]:
MODELOS = [
    "google/gemma-4-31b-it:free",
    "nvidia/nemotron-3.5-lightning:free",
    "cohere/north-mini-code:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "xingchen-agi/xing4.0-29b-a4b:free",
]

OPENCODE_MODELOS = [
    "deepseek-v4.1-flash",
    "claude-haiku-5-5",
    "glm-5.3-flash",
]


def listar_modelos_free():
    """Descobre dinamicamente os modelos ':free' disponiveis na OpenRouter.

    A OpenRouter rotaciona os modelos gratuitos; fixar slugs quebra com o tempo.
    """
    import urllib.request
    try:
        with urllib.request.urlopen(
            "https://openrouter.ai/api/v1/models", timeout=30
        ) as resp:
            dados = json.loads(resp.read().decode("utf-8"))
        return [
            m["id"] for m in dados.get("data", [])
            if str(m.get("id", "")).endswith(":free")
        ]
    except Exception as e:
        print("Nao foi possivel listar modelos free:", e)
        return []


class LLMOpenRouter:
    def __init__(self, api_key, modelos=None, temperature=0.0, timeout=60):
        from openai import OpenAI
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key or "sk-dummy",
        )
        self.modelos = list(modelos or MODELOS)
        self.temperature = temperature
        self.timeout = timeout
        self._free_cache = None
        self.chamadas = []
        self.uso = []

    def _candidatos(self):
        if getattr(self, "_free_cache", None) is None:
            self._free_cache = listar_modelos_free()
        candidatos = list(self.modelos)
        for modelo in self._free_cache:
            if modelo not in candidatos:
                candidatos.append(modelo)
        return candidatos[:12]

    def chat(self, mensagens, agente=None):
        ultimo = None
        tentativas = 0
        for modelo in self._candidatos():
            tentativas += 1
            try:
                resp = self.client.chat.completions.create(
                    model=modelo,
                    messages=mensagens,
                    temperature=self.temperature,
                    timeout=self.timeout,
                    max_tokens=1200,
                )
                texto = resp.choices[0].message.content or ""
                self.chamadas.append({"agente": agente, "modelo": modelo})
                if getattr(resp, "usage", None) is not None:
                    self.uso.append({
                        "agente": agente,
                        "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                        "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    })
                return texto
            except Exception as e:
                ultimo = e
        raise RuntimeError(
            "Todos os modelos falharam (" + str(tentativas)
            + " tentativas). Ultimo erro: " + str(ultimo)
        )


class LLMOpenCodeZen:
    """Cliente para OpenCode Zen (chaves `oc_sk_...`).

    O endpoint e compativel com OpenAI Chat Completions. A lista padrao usa
    modelos com boa aderencia a JSON e fallback entre eles.
    """

    def __init__(self, api_key, modelos=None, temperature=0.0, timeout=75):
        from openai import OpenAI
        self.client = OpenAI(
            base_url="https://opencode.ai/zen/v1",
            api_key=api_key,
            timeout=timeout,
            default_headers={"User-Agent": "opencode/1.19.4"},
        )
        self.modelos = list(modelos or OPENCODE_MODELOS)
        self.temperature = temperature
        self.timeout = timeout
        self.chamadas = []
        self.uso = []

    def chat(self, mensagens, agente=None):
        ultimo = None
        tentativas = 0
        for modelo in self.modelos:
            tentativas += 1
            try:
                resp = self.client.chat.completions.create(
                    model=modelo,
                    messages=mensagens,
                    temperature=self.temperature,
                    timeout=self.timeout,
                    max_tokens=1200,
                )
                texto = resp.choices[0].message.content or ""
                self.chamadas.append({"agente": agente, "modelo": modelo})
                if getattr(resp, "usage", None) is not None:
                    self.uso.append({
                        "agente": agente,
                        "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                        "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    })
                return texto
            except Exception as e:
                ultimo = e
        raise RuntimeError(
            "Todos os modelos OpenCode Zen falharam (" + str(tentativas)
            + " tentativas). Ultimo erro: " + str(ultimo)
        )


def criar_llm(chave):
    chave = (chave or "").strip()
    if chave.startswith("oc_sk_"):
        return LLMOpenCodeZen(chave)
    return LLMOpenRouter(chave)


class MockLLM:
    """LLM simulado com roteiro por agente.

    `roteiro` mapeia o nome do agente para uma lista de respostas. Quando a lista
    termina, levanta erro para deixar claro que o teste nao cobriu uma chamada.
    """

    def __init__(self, roteiro=None):
        self.roteiro = {
            agente: list(respostas)
            for agente, respostas in (roteiro or {}).items()
        }
        self.chamadas = []

    def chat(self, mensagens, agente=None):
        self.chamadas.append({"agente": agente, "mensagens": list(mensagens)})
        fila = self.roteiro.get(agente)
        if fila:
            return fila.pop(0)
        fila = self.roteiro.get("*")
        if fila:
            return fila.pop(0)
        raise RuntimeError("MockLLM sem resposta para agente=" + str(agente))

In [7]:
def parse_json(texto):
    t = (texto or "").strip()
    if t.startswith("```"):
        t = t.strip("`").strip()
        if t.lower().startswith("json"):
            t = t[4:].strip()
    inicio = t.find("{")
    fim = t.rfind("}")
    if inicio == -1 or fim == -1 or fim <= inicio:
        return None
    try:
        return json.loads(t[inicio:fim + 1])
    except Exception:
        return None


def _extrair_payload(obj):
    if isinstance(obj, dict) and isinstance(obj.get("final"), dict):
        return obj["final"]
    return obj


def chamar_json(llm, agente, system_prompt, tarefa, schema, tentativas=2):
    """Chama o LLM e valida o payload contra um schema Pydantic.

    Retorna (modelo, bruto, erro). Em caso de falha, o chamador usa um fallback
    deterministico, preservando a robustez do sistema.
    """
    mensagens = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": json.dumps(tarefa, ensure_ascii=False, default=str),
        },
    ]
    bruto = ""
    ultimo_erro = None
    for _ in range(tentativas):
        try:
            bruto = llm.chat(mensagens, agente=agente)
        except Exception as e:
            return None, bruto, "falha_llm: " + str(e)
        obj = parse_json(bruto)
        if obj is None:
            ultimo_erro = "json_invalido"
        else:
            try:
                return schema.model_validate(_extrair_payload(obj)), bruto, None
            except ValidationError as e:
                ultimo_erro = "contrato_invalido: " + str(e)
        mensagens.append({"role": "assistant", "content": bruto})
        mensagens.append({
            "role": "user",
            "content": (
                "Sua resposta violou o contrato. Corrija e responda APENAS com JSON "
                "valido. Erro: " + str(ultimo_erro)
            ),
        })
    return None, bruto, ultimo_erro


def loop_react(llm, agente, system_prompt, tarefa, nomes_ferramentas,
               schema_final, max_passos=MAX_PASSOS_ESPECIALISTA):
    """Ciclo decisao -> acao -> observacao com contrato de saida.

    Retorna um dicionario com:
    - final: payload validado ou None
    - tool_calls: ferramentas chamadas
    - ultimo_resultado: retorno da ultima ferramenta
    - ultimo_sucesso: retorno bem-sucedido mais recente
    - erro: motivo de encerramento sem contrato
    """
    mensagens = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": json.dumps(tarefa, ensure_ascii=False, default=str),
        },
    ]
    tool_calls = []
    ultimo_resultado = None
    ultimo_sucesso = None
    ultimo_erro = None

    for passo in range(max_passos):
        try:
            bruto = llm.chat(mensagens, agente=agente)
        except Exception as e:
            return {
                "final": None, "tool_calls": tool_calls,
                "ultimo_resultado": ultimo_resultado,
                "ultimo_sucesso": ultimo_sucesso,
                "erro": "falha_llm: " + str(e), "passos": passo + 1,
            }

        obj = parse_json(bruto)
        if obj is None:
            mensagens.append({"role": "assistant", "content": bruto})
            mensagens.append({
                "role": "user",
                "content": "Formato invalido. Responda apenas com JSON.",
            })
            ultimo_erro = "json_invalido"
            continue

        if "final" in obj:
            try:
                final = schema_final.model_validate(obj["final"])
            except ValidationError as e:
                mensagens.append({"role": "assistant", "content": bruto})
                mensagens.append({
                    "role": "user",
                    "content": (
                        "O payload final violou o contrato. Corrija. Erro: " + str(e)
                    ),
                })
                ultimo_erro = "contrato_invalido"
                continue
            return {
                "final": final.model_dump(),
                "tool_calls": tool_calls,
                "ultimo_resultado": ultimo_resultado,
                "ultimo_sucesso": ultimo_sucesso,
                "erro": None,
                "passos": passo + 1,
            }

        acao = obj.get("acao") or obj.get("action")
        argumentos = obj.get("argumentos") or obj.get("action_input") or {}
        if acao not in nomes_ferramentas:
            mensagens.append({"role": "assistant", "content": bruto})
            mensagens.append({
                "role": "user",
                "content": (
                    "Acao invalida ou ausente. Use uma destas: "
                    + ", ".join(nomes_ferramentas)
                    + ". Ou retorne {\"final\": {...}}."
                ),
            })
            ultimo_erro = "acao_invalida"
            continue

        resultado = executar_ferramenta(acao, argumentos)
        tool_calls.append(acao)
        ultimo_resultado = resultado
        if resultado.get("ok"):
            ultimo_sucesso = resultado
        mensagens.append({"role": "assistant", "content": bruto})
        mensagens.append({
            "role": "user",
            "content": "Observacao: " + json.dumps(resultado, ensure_ascii=False),
        })

    return {
        "final": None,
        "tool_calls": tool_calls,
        "ultimo_resultado": ultimo_resultado,
        "ultimo_sucesso": ultimo_sucesso,
        "erro": ultimo_erro or "limite_de_passos",
        "passos": max_passos,
    }


print("Camada LLM pronta.")

Camada LLM pronta.


## 7. Fase 1 (C2) — o agente único dentro do LangGraph

Antes de decompor, reorganizamos o agente da Entrega I como um grafo. A lógica ReAct
é a mesma; o que muda é a existência de estado explícito, checkpoint e trilha de
auditoria. Essa configuração isola o ganho da orquestração.

In [8]:
EXEMPLO_ACAO = json.dumps({
    "thought": "preciso ver o schema",
    "action": "inspecionar_schema",
    "action_input": {"tabela": "transacoes"},
})
EXEMPLO_FINAL = json.dumps({
    "thought": "ja tenho a resposta",
    "final_answer": "resposta ao usuario",
})

PROMPT_SISTEMA_UNICO = (
    "Voce e um analista de dados do Banco Aurora. Responda perguntas de negocio "
    "consultando o data warehouse com as ferramentas disponiveis.\n\n"
    "Ferramentas:\n" + descricao_ferramentas() + "\n\n"
    "Responda SEMPRE com um unico objeto JSON, sem texto fora dele, em um destes "
    "formatos:\n" + EXEMPLO_ACAO + "\n" + EXEMPLO_FINAL + "\n\n"
    "Regras: nunca invente numeros; se a informacao nao existir no DW, diga que nao "
    "ha dados ou use escalar_analista; nunca emita comandos que nao sejam SELECT; "
    "seja conciso na resposta final."
)


def rodar_agente_unico(pergunta, llm, verbose=False):
    """Loop ReAct da Entrega I, mantido para a comparacao C1/C2."""
    try:
        pergunta = validar_entrada(pergunta)
    except ValueError as e:
        return {
            "resposta": "Entrada invalida: " + str(e),
            "trace": [], "tool_calls": [], "steps": 0,
        }

    mensagens = [
        {"role": "system", "content": PROMPT_SISTEMA_UNICO},
        {"role": "user", "content": pergunta},
    ]
    trace = []
    tool_calls = []
    falhas_sql = 0

    for passo in range(1, MAX_STEPS + 1):
        try:
            bruto = llm.chat(mensagens, agente="unico")
        except Exception as e:
            trace.append({"passo": passo, "erro_llm": str(e)})
            return {
                "resposta": "Nao consegui consultar o modelo agora. Caso encaminhado a um analista.",
                "trace": trace, "tool_calls": tool_calls, "steps": passo,
            }

        decisao = parse_json(bruto)
        if decisao is None:
            trace.append({"passo": passo, "erro": "formato_invalido"})
            mensagens.append({"role": "assistant", "content": bruto})
            mensagens.append({
                "role": "user",
                "content": "Formato invalido. Responda apenas com o objeto JSON esperado.",
            })
            continue

        if decisao.get("final_answer"):
            trace.append({
                "passo": passo, "thought": decisao.get("thought", ""), "final": True,
            })
            return {
                "resposta": decisao["final_answer"],
                "trace": trace, "tool_calls": tool_calls, "steps": passo,
            }

        nome = decisao.get("action")
        entrada = decisao.get("action_input") or {}
        if not nome:
            trace.append({"passo": passo, "erro": "sem_acao"})
            mensagens.append({
                "role": "user", "content": "Faltou a chave action ou final_answer.",
            })
            continue

        resultado = executar_ferramenta(nome, entrada)
        tool_calls.append(nome)
        obs = json.dumps(resultado, ensure_ascii=False)
        trace.append({
            "passo": passo, "thought": decisao.get("thought", ""),
            "action": nome, "ok": resultado.get("ok"),
            "observacao": obs[:280],
        })
        if verbose:
            print("passo", passo, "->", nome, "| ok =", resultado.get("ok"))

        mensagens.append({"role": "assistant", "content": json.dumps(decisao, ensure_ascii=False)})
        mensagens.append({"role": "user", "content": "Observacao: " + obs})

        if nome == "executar_sql" and not resultado.get("ok"):
            falhas_sql += 1
            if falhas_sql > MAX_SQL_RETRIES:
                executar_ferramenta("escalar_analista", {"motivo": "falhas repetidas de SQL"})
                tool_calls.append("escalar_analista")
                trace.append({"passo": passo, "recuperacao": "escalonado apos falhas de SQL"})
                return {
                    "resposta": "Nao consegui responder com os dados disponiveis. Encaminhei o caso a um analista.",
                    "trace": trace, "tool_calls": tool_calls, "steps": passo,
                }

    trace.append({"passo": MAX_STEPS, "erro": "limite_de_passos"})
    return {
        "resposta": "Atingi o limite de passos sem concluir. Caso encaminhado a um analista.",
        "trace": trace, "tool_calls": tool_calls, "steps": MAX_STEPS,
    }


def node_validar_unico(state):
    try:
        pergunta = validar_entrada(state["pergunta"])
    except ValueError as e:
        env = novo_envelope(
            "sistema", "coordenador", "entrada_invalida",
            {"erro": str(e)}, state["correlation_id"],
        )
        return {
            "status": "invalido",
            "resposta": "Entrada invalida: " + str(e),
            "erro": str(e),
            "mensagens": [env],
            "auditoria": [entrada_auditoria(env)],
        }
    env = novo_envelope(
        "sistema", "coordenador", "entrada_validada",
        {"pergunta": pergunta}, state["correlation_id"],
    )
    return {
        "pergunta": pergunta,
        "status": "validado",
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


def node_agente_unico(state, llm):
    resultado = rodar_agente_unico(state["pergunta"], llm)
    env = novo_envelope(
        "agente_unico", "coordenador", "resultado_agente_unico",
        {
            "resposta": resultado["resposta"],
            "trace": resultado["trace"],
            "tool_calls": resultado["tool_calls"],
        },
        state["correlation_id"],
        proveniencia=["entrega_I"],
    )
    return {
        "resposta": resultado["resposta"],
        "trace": resultado["trace"],
        "tool_calls": resultado["tool_calls"],
        "status": "concluido",
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


def node_finalizar_unico(state):
    env = novo_envelope(
        "agente_unico", "usuario", "resposta_final",
        {
            "resposta": state.get("resposta", ""),
            "status": state.get("status", "concluido"),
        },
        state["correlation_id"],
        proveniencia=["resultado_agente_unico"],
    )
    return {
        "status": state.get("status", "concluido"),
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


def construir_grafo_unico(llm):
    builder = StateGraph(EstadoAurora)
    builder.add_node("validar", node_validar_unico)
    builder.add_node("agente", lambda s: node_agente_unico(s, llm))
    builder.add_node("finalizar", node_finalizar_unico)
    builder.add_edge(START, "validar")
    builder.add_conditional_edges(
        "validar",
        lambda s: "agente" if s.get("status") != "invalido" else "finalizar",
        {"agente": "agente", "finalizar": "finalizar"},
    )
    builder.add_edge("agente", "finalizar")
    builder.add_edge("finalizar", END)
    return builder.compile(checkpointer=InMemorySaver())


def estado_inicial(pergunta, thread_id):
    return {
        "pergunta": pergunta,
        "correlation_id": novo_id("corr"),
        "thread_id": thread_id,
        "mensagens": [],
        "auditoria": [],
        "tool_calls": [],
        "revisoes": 0,
        "tentativas_sql": 0,
        "status": "iniciado",
    }


def executar_c1(pergunta, llm):
    return rodar_agente_unico(pergunta, llm)


def executar_c2(pergunta, llm, thread_id=None):
    thread_id = thread_id or novo_id("thread-c2")
    grafo = construir_grafo_unico(llm)
    return grafo.invoke(
        estado_inicial(pergunta, thread_id),
        config={"configurable": {"thread_id": thread_id}},
    )


print("Fase 1 (C2) pronta: grafo do agente unico com checkpoint e auditoria.")

Fase 1 (C2) pronta: grafo do agente unico com checkpoint e auditoria.


## 8. Fase 2 (C3) — os quatro agentes

Cada agente tem um prompt focado, um subconjunto de ferramentas e um contrato de
saída. O Coordenador é o único que fala com o usuário; os especialistas conversam
por envelopes.

In [9]:
PROMPT_COORDENADOR_PLANO = (
    "Voce e o Coordenador de um sistema multiagente de BI do Banco Aurora.\n"
    "Sua funcao e interpretar a pergunta, classificar o tipo e planejar. Voce NAO "
    "executa SQL, NAO inspeciona o schema e NAO responde dados.\n\n"
    "Catalogo disponivel:\n" + CATALOGO + "\n\n"
    "Glossario disponivel: " + ", ".join(GLOSSARIO.keys()) + "\n\n"
    "Classifique o tipo:\n"
    "- 'definicao': pergunta sobre o significado de uma metrica/regra.\n"
    "- 'consulta': pergunta que exige dados do DW.\n"
    "- 'fora_escopo': pedido destrutivo, fora do DW ou sem dados.\n"
    "- 'invalido': pergunta vazia ou ininteligivel.\n\n"
    "Responda APENAS com JSON neste schema:\n"
    '{"tipo":"consulta|definicao|fora_escopo|invalido",'
    '"objetivo":"o que precisa ser respondido",'
    '"termos":["metricas ou termos de negocio"],'
    '"periodo":"periodo explicito ou null",'
    '"entidades":["clientes, tabelas, agrupamentos"],'
    '"precisa_dominio":true,'
    '"risco":"baixo|medio|alto",'
    '"justificativa":"por que essa classificacao"}\n'
    "Use risco alto para pedidos destrutivos, dados sensiveis ou escopo ambiguo."
)

PROMPT_COORDENADOR_DECIDE = (
    "Voce e o Coordenador do sistema multiagente de BI do Banco Aurora.\n"
    "Voce recebe a pergunta, o plano, o brief de dominio, o artefato de consulta e o "
    "parecer do auditor. Decida o proximo passo.\n\n"
    "Acoes possiveis:\n"
    "- 'finalizar': o parecer aprovou e ha evidencia suficiente. Forneca a resposta "
    "direta ao usuario, sem inventar numeros.\n"
    "- 'revisar': o parecer reprovou e ainda ha orcamento. Forneca uma instrucao "
    "especifica ao Analista SQL.\n"
    "- 'escalar': risco alto, falhas repetidas ou orcamento esgotado.\n"
    "- 'esclarecer': falta definicao essencial para responder com seguranca.\n\n"
    "Responda APENAS com JSON:\n"
    '{"acao":"finalizar|revisar|escalar|esclarecer",'
    '"resposta":"resposta ao usuario ou null",'
    '"instrucao":"instrucao ao analista ou null",'
    '"motivo":"justificativa da decisao"}'
)

PROMPT_DOMINIO = (
    "Voce e o Especialista de Dominio do Banco Aurora. Sua funcao e resolver a "
    "semantica de negocio: definicoes, metricas, tabelas, colunas e regras. Voce NAO "
    "escreve SQL e NAO responde ao usuario.\n\n"
    "Ferramentas disponiveis:\n"
    "- inspecionar_schema: lista tabelas ou colunas de uma tabela.\n"
    "- buscar_documentacao: busca definicoes no glossario.\n\n"
    "Fluxo: use as ferramentas para confirmar tabelas/colunas e definicoes. Depois "
    "retorne o brief. Se nao houver definicao para um termo essencial, registre em "
    "'lacunas' e reduza a confianca.\n\n"
    "Formato de acao:\n"
    '{"acao":"inspecionar_schema","argumentos":{"tabela":"transacoes"}}\n'
    "Formato final:\n"
    '{"final":{"definicoes":{"termo":"definicao"},'
    '"tabelas":["tabela"],"colunas":{"tabela":["coluna"]},'
    '"regras":["regra de negocio"],"lacunas":[],"confianca":0.0}}'
)

PROMPT_ANALISTA = (
    "Voce e o Analista SQL do Banco Aurora. Sua funcao e formular, executar e corrigir "
    "uma consulta SELECT somente leitura para responder a pergunta. Voce recebe o "
    "brief de dominio e, quando houver, o feedback do auditor.\n\n"
    "Regras: use apenas SELECT; respeite as definicoes e regras do brief; filtre o "
    "periodo quando existir; use agregacao quando a pergunta pedir contagem, total, "
    "soma ou media; nunca invente o resultado — o sistema anexa o retorno real da "
    "ultima execucao bem-sucedida.\n\n"
    "Formato de acao:\n"
    '{"acao":"executar_sql","argumentos":{"sql":"SELECT ..."}}\n'
    "Formato final (depois de executar):\n"
    '{"final":{"sql":"SELECT ...","explicacao":"o que a consulta faz",'
    '"hipoteses":["hipotese relevante"]}}'
)

PROMPT_AUDITOR = (
    "Voce e o Auditor de Resultados, independente do Analista SQL. Voce NAO escreve "
    "uma nova consulta para responder ao usuario e NAO toma decisoes de negocio. Sua "
    "funcao e verificar evidencia.\n\n"
    "Verifique: (1) a SQL responde a pergunta? (2) usa as definicoes/regras do brief? "
    "(3) o periodo, filtros e agregacao estao corretos? (4) o resultado e vazio, "
    "absurdo ou incompativel? (5) ha erro de SQL? Considere as checagens "
    "deterministicas fornecidas.\n\n"
    "Aprove somente se houver evidencia suficiente. Se reprovar, forneca uma correcao "
    "especifica e acionavel ao Analista SQL.\n\n"
    "Responda APENAS com JSON:\n"
    '{"aprovado":true,"motivos":["motivo"],"problemas":["problema"],'
    '"correcao_sugerida":"instrucao ou null","confianca":0.0}'
)

In [10]:
def _plano_fallback(pergunta: str) -> PlanoTarefa:
    t = _normalizar(pergunta)
    if detectar_intencao_destrutiva(pergunta):
        return PlanoTarefa(
            tipo="fora_escopo", objetivo=pergunta, risco="alto",
            justificativa="Pedido destrutivo detectado por guardrail deterministico.",
        )
    if any(p in t for p in ["definicao", "definição", "significa", "o que e", "o que é"]):
        return PlanoTarefa(
            tipo="definicao", objetivo=pergunta, risco="baixo",
            justificativa="Fallback deterministico: pergunta sobre definicao.",
        )
    return PlanoTarefa(
        tipo="consulta", objetivo=pergunta, risco="baixo",
        justificativa="Fallback deterministico: pergunta de consulta.",
    )


def _brief_fallback(pergunta: str) -> BriefDominio:
    t = _normalizar(pergunta)
    definicoes = {k: v for k, v in GLOSSARIO.items() if k in t}
    metricas = ["churn", "ticket medio", "limite", "margem", "lucro", "receita", "inadimplencia"]
    lacunas = []
    if not definicoes and any(m in t for m in metricas):
        lacunas.append(
            "Nenhuma definicao de metrica foi encontrada no glossario para a pergunta."
        )
    tabelas = [
        r[0] for r in DW.execute(
            "SELECT name FROM sqlite_master WHERE type = ? ORDER BY name", ("table",)
        ).fetchall()
    ]
    confianca = 0.6 if definicoes or not lacunas else 0.25
    return BriefDominio(
        definicoes=definicoes,
        tabelas=tabelas,
        colunas={},
        regras=["Consultar apenas tabelas do catalogo e respeitar o periodo informado."],
        lacunas=lacunas,
        confianca=confianca,
    )


def checagens_deterministicas(plano: dict, brief: dict, artefato: dict) -> dict:
    criticos = []
    avisos = []
    sql = _normalizar(artefato.get("sql", ""))
    resultado = artefato.get("resultado") or {}

    if artefato.get("erro"):
        criticos.append("erro_sql: " + str(artefato["erro"]))
    if sql and not sql.lstrip().startswith("select"):
        criticos.append("sql_nao_select")
    if not sql:
        criticos.append("sql_ausente")

    periodo = _normalizar(plano.get("periodo") or "")
    if periodo:
        tokens_periodo = [t for t in re.findall(r"\d{4}-\d{2}", periodo)]
        if "marco" in periodo and not tokens_periodo:
            tokens_periodo = ["2026-03"]
        if tokens_periodo and not any(t in sql for t in tokens_periodo):
            criticos.append("periodo_ausente_no_sql")
        elif not tokens_periodo and "marco" not in sql:
            criticos.append("periodo_ausente_no_sql")

    objetivo = _normalizar(plano.get("objetivo", ""))
    if any(x in objetivo for x in ["quantas", "quantos", "total", "soma", "media", "média"]):
        if not any(f in sql for f in ["count(", "sum(", "avg("]):
            criticos.append("agregacao_ausente_no_sql")

    tabelas_brief = [t.lower() for t in brief.get("tabelas", []) if t]
    if tabelas_brief and not any(t in sql for t in tabelas_brief):
        criticos.append("tabela_do_brief_ausente_no_sql")

    if int(resultado.get("n", 0)) == 0:
        avisos.append("resultado_vazio")

    return {"criticos": criticos, "avisos": avisos}

In [11]:
def node_validar_multi(state):
    try:
        pergunta = validar_entrada(state["pergunta"])
    except ValueError as e:
        env = novo_envelope(
            "sistema", "coordenador", "entrada_invalida",
            {"erro": str(e)}, state["correlation_id"],
        )
        return {
            "status": "invalido",
            "resposta": "Entrada invalida: " + str(e),
            "erro": str(e),
            "mensagens": [env],
            "auditoria": [entrada_auditoria(env)],
        }
    env = novo_envelope(
        "sistema", "coordenador", "entrada_validada",
        {"pergunta": pergunta}, state["correlation_id"],
    )
    return {
        "pergunta": pergunta,
        "status": "validado",
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


def node_coordenador_planeja(state, llm):
    pergunta = state["pergunta"]

    if detectar_intencao_destrutiva(pergunta):
        plano = PlanoTarefa(
            tipo="fora_escopo", objetivo=pergunta, risco="alto",
            justificativa=(
                "Pedido destrutivo bloqueado antes de qualquer execucao SQL. "
                "O sistema e somente leitura."
            ),
        )
    else:
        tarefa = {
            "pergunta": pergunta,
            "catalogo": CATALOGO,
            "glossario": list(GLOSSARIO.keys()),
            "instrucao": "Planeje sem executar SQL.",
        }
        modelo, _, erro = chamar_json(
            llm, "coordenador_planeja", PROMPT_COORDENADOR_PLANO,
            tarefa, PlanoTarefa,
        )
        plano = modelo or _plano_fallback(pergunta)
        if modelo is None:
            plano.justificativa += " (fallback: " + str(erro) + ")"

    destino = "usuario" if plano.tipo == "fora_escopo" else "especialista_dominio"
    pacote = empacotar(
        state, "coordenador", destino, "plano_tarefa",
        plano.model_dump(), proveniencia=["pergunta"],
    )
    return {
        "plano": plano.model_dump(),
        "status": "planejado",
        **pacote,
    }


def node_dominio(state, llm):
    plano = state.get("plano", {})
    tarefa = {
        "pergunta": state["pergunta"],
        "plano": plano,
        "catalogo": CATALOGO,
        "glossario": GLOSSARIO,
        "instrucao": (
            "Resolva a semantica. Nao escreva SQL. Se o termo essencial nao estiver "
            "no glossario, registre a lacuna e reduza a confianca."
        ),
    }
    resultado = loop_react(
        llm, "especialista_dominio", PROMPT_DOMINIO, tarefa,
        ["inspecionar_schema", "buscar_documentacao"],
        BriefDominio,
    )
    if resultado["final"] is not None:
        brief = BriefDominio.model_validate(resultado["final"])
    else:
        brief = _brief_fallback(state["pergunta"])

    status = "dominio_resolvido"
    if brief.lacunas and brief.confianca < 0.5:
        status = "dominio_incompleto"

    pacote = empacotar(
        state, "especialista_dominio", "coordenador", "brief_dominio",
        brief.model_dump(), proveniencia=["plano_tarefa"],
    )
    return {
        "brief": brief.model_dump(),
        "status": status,
        "tool_calls": resultado.get("tool_calls", []),
        **pacote,
    }


def node_politica(state):
    plano = state.get("plano", {})
    if plano.get("risco") == "alto":
        env = novo_envelope(
            "coordenador", "usuario", "politica_bloqueio",
            {
                "motivo": "Plano classificado como risco alto; requer aprovacao humana.",
                "plano": plano,
            },
            state["correlation_id"],
            proveniencia=["plano_tarefa"],
        )
        return {
            "status": "politica_bloqueada",
            "erro": "Risco alto: aprovacao humana necessaria antes da consulta.",
            "mensagens": [env],
            "auditoria": [entrada_auditoria(env)],
        }

    env = novo_envelope(
        "coordenador", "analista_sql", "politica_ok",
        {
            "restricoes": {
                "somente_leitura": True,
                "mascarar_pii": True,
                "max_linhas": MAX_ROWS,
            }
        },
        state["correlation_id"],
        proveniencia=["plano_tarefa", "brief_dominio"],
    )
    return {
        "status": "politica_ok",
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


def node_analista(state, llm):
    feedback = (state.get("decisao") or {}).get("instrucao")
    tarefa = {
        "pergunta": state["pergunta"],
        "plano": state.get("plano", {}),
        "brief": state.get("brief", {}),
        "feedback_do_auditor": feedback,
        "tentativa_de_revisao": state.get("revisoes", 0),
        "instrucao": (
            "Formule e execute SQL. Se o auditor enviou feedback, corrija exatamente "
            "o problema apontado. Nao invente o resultado."
        ),
    }
    resultado = loop_react(
        llm, "analista_sql", PROMPT_ANALISTA, tarefa,
        ["executar_sql"], ArtefatoParcial,
        max_passos=MAX_TENTATIVAS_SQL + 2,
    )

    final = resultado.get("final") or {}
    ultimo_sucesso = resultado.get("ultimo_sucesso")
    ultimo_resultado = resultado.get("ultimo_resultado")

    if ultimo_sucesso:
        dados = ultimo_sucesso.get("data", {})
    else:
        dados = {"columns": [], "rows": [], "n": 0}

    sql_final = (
        (dados.get("sql") if ultimo_sucesso else None)
        or final.get("sql")
        or ((ultimo_resultado or {}).get("data") or {}).get("sql", "")
    )
    if ultimo_sucesso:
        erro = None
    elif ultimo_resultado and not ultimo_resultado.get("ok"):
        erro = ultimo_resultado.get("error")
    else:
        erro = resultado.get("erro") or "Nao foi possivel executar uma consulta valida."

    explicacao = final.get("explicacao") or (
        "Consulta executada sem contrato final explicito."
        if ultimo_sucesso else "Nao foi possivel concluir a consulta dentro do orcamento."
    )
    n_sql = sum(1 for t in resultado.get("tool_calls", []) if t == "executar_sql")

    artefato = ArtefatoConsulta(
        sql=sql_final or "",
        resultado=dados,
        explicacao=explicacao,
        hipoteses=list(final.get("hipoteses") or []),
        erro=erro,
        tentativas=max(n_sql, 1),
        tool_calls=resultado.get("tool_calls", []),
    )

    pacote = empacotar(
        state, "analista_sql", "auditor", "artefato_consulta",
        artefato.model_dump(), proveniencia=["brief_dominio", "politica_ok"],
    )
    return {
        "artefato": artefato.model_dump(),
        "tentativas_sql": state.get("tentativas_sql", 0) + n_sql,
        "tool_calls": resultado.get("tool_calls", []),
        "status": "consulta_executada",
        **pacote,
    }


def node_auditor(state, llm):
    artefato = state.get("artefato", {})
    brief = state.get("brief", {})
    plano = state.get("plano", {})
    checagens = checagens_deterministicas(plano, brief, artefato)

    tarefa = {
        "pergunta": state["pergunta"],
        "plano": plano,
        "brief": brief,
        "artefato": artefato,
        "checagens_deterministicas": checagens,
        "revisoes_anteriores": state.get("revisoes", 0),
        "instrucao": (
            "Aprove apenas com evidencia suficiente. Se reprovar, diga exatamente o "
            "que o analista deve corrigir."
        ),
    }
    modelo, _, erro = chamar_json(
        llm, "auditor", PROMPT_AUDITOR, tarefa, ParecerVerificacao,
    )

    if modelo is None:
        aprovado = not checagens["criticos"] and not artefato.get("erro")
        modelo = ParecerVerificacao(
            aprovado=aprovado,
            motivos=["Fallback deterministico do auditor."],
            problemas=checagens["criticos"] + checagens["avisos"],
            correcao_sugerida=(
                "; ".join(checagens["criticos"]) if not aprovado else None
            ),
            confianca=0.5,
        )
        if erro:
            modelo.motivos.append("LLM indisponivel: " + str(erro))

    # Guardrail: o auditor nao pode aprovar um artefato com erro critico.
    if checagens["criticos"] and modelo.aprovado:
        modelo.aprovado = False
        modelo.motivos.append(
            "Override deterministico: havia checagem critica mesmo com parecer favoravel."
        )
        modelo.problemas = list(dict.fromkeys(modelo.problemas + checagens["criticos"]))
        modelo.correcao_sugerida = modelo.correcao_sugerida or "; ".join(checagens["criticos"])

    pacote = empacotar(
        state, "auditor", "coordenador", "parecer_verificacao",
        modelo.model_dump(), proveniencia=["artefato_consulta"],
    )
    return {
        "parecer": modelo.model_dump(),
        "status": "auditado",
        **pacote,
    }

In [12]:
def _resposta_de_artefato(artefato: dict) -> str:
    resultado = artefato.get("resultado") or {}
    linhas = resultado.get("rows") or []
    if len(linhas) == 1 and len(linhas[0]) == 1:
        valor = list(linhas[0].values())[0]
        return "O resultado da consulta e: " + str(valor) + "."
    return "A consulta retornou " + str(resultado.get("n", 0)) + " linha(s)."


def _decisao_fallback(state) -> DecisaoCoordenador:
    parecer = state.get("parecer") or {}
    artefato = state.get("artefato") or {}
    revisoes = state.get("revisoes", 0)
    if parecer.get("aprovado") and not artefato.get("erro"):
        return DecisaoCoordenador(
            acao="finalizar",
            resposta=_resposta_de_artefato(artefato),
            motivo="Fallback: parecer aprovado.",
        )
    if revisoes < MAX_REVISOES:
        return DecisaoCoordenador(
            acao="revisar",
            instrucao=parecer.get("correcao_sugerida")
            or "Reexamine a consulta e corrija os problemas apontados.",
            motivo="Fallback: parecer reprovado com orcamento de revisao.",
        )
    return DecisaoCoordenador(
        acao="escalar",
        motivo="Fallback: orcamento de revisao esgotado.",
    )


def node_coordenador_decide(state, llm):
    plano = state.get("plano", {})
    brief = state.get("brief", {})
    artefato = state.get("artefato", {})
    parecer = state.get("parecer", {})
    revisoes = state.get("revisoes", 0)

    tarefa = {
        "pergunta": state["pergunta"],
        "plano": plano,
        "brief": brief,
        "artefato": artefato,
        "parecer": parecer,
        "revisoes_usadas": revisoes,
        "max_revisoes": MAX_REVISOES,
        "instrucao": (
            "Decida com base na evidencia. Nao finalize se o parecer reprovou. "
            "Nao invente numeros."
        ),
    }
    modelo, _, erro = chamar_json(
        llm, "coordenador_decide", PROMPT_COORDENADOR_DECIDE,
        tarefa, DecisaoCoordenador,
    )
    decisao = modelo or _decisao_fallback(state)
    if modelo is None:
        decisao.motivo += " (fallback: " + str(erro) + ")"

    # Governanca de decisao: o coordenador nao pode ignorar o auditor.
    if decisao.acao == "finalizar" and not parecer.get("aprovado"):
        decisao.acao = "revisar" if revisoes < MAX_REVISOES else "escalar"
        decisao.motivo += " Override: parecer reprovado."
    if decisao.acao == "revisar" and revisoes >= MAX_REVISOES:
        decisao.acao = "escalar"
        decisao.motivo += " Override: limite de revisoes atingido."
    if decisao.acao == "finalizar" and not decisao.resposta:
        decisao.resposta = _resposta_de_artefato(artefato)
    if decisao.acao == "finalizar":
        resultado = artefato.get("resultado") or {}
        linhas = resultado.get("rows") or []
        if len(linhas) == 1 and len(linhas[0]) == 1:
            valor = str(list(linhas[0].values())[0])
            if valor not in (decisao.resposta or ""):
                decisao.resposta = (
                    (decisao.resposta or "")
                    + " Resultado da consulta: " + valor + "."
                ).strip()

    novas_revisoes = revisoes + (1 if decisao.acao == "revisar" else 0)
    destino = {
        "finalizar": "usuario",
        "revisar": "analista_sql",
        "escalar": "usuario",
        "esclarecer": "usuario",
    }[decisao.acao]
    pacote = empacotar(
        state, "coordenador", destino, "decisao_" + decisao.acao,
        decisao.model_dump(), proveniencia=["plano_tarefa", "parecer_verificacao"],
    )
    return {
        "decisao": decisao.model_dump(),
        "revisoes": novas_revisoes,
        "status": "decidido",
        **pacote,
    }


def _formatar_evidencias(state) -> list[str]:
    evidencias = []
    brief = state.get("brief") or {}
    artefato = state.get("artefato") or {}
    parecer = state.get("parecer") or {}
    plano = state.get("plano") or {}

    if brief.get("definicoes"):
        itens = "; ".join(k + ": " + v for k, v in brief["definicoes"].items())
        evidencias.append("Definicoes aplicadas: " + itens)
    if brief.get("regras"):
        evidencias.append("Regras aplicadas: " + "; ".join(brief["regras"]))
    if artefato.get("sql"):
        evidencias.append("SQL executado: " + artefato["sql"])
    resultado = artefato.get("resultado") or {}
    if resultado:
        evidencias.append(
            "Evidencia: " + str(resultado.get("n", 0)) + " linha(s); colunas: "
            + ", ".join(resultado.get("columns", []))
        )
        linhas = resultado.get("rows") or []
        if linhas:
            evidencias.append(
                "Amostra: " + json.dumps(linhas[:5], ensure_ascii=False)
            )
    if parecer.get("problemas"):
        evidencias.append("Ressalvas do auditor: " + "; ".join(parecer["problemas"]))
    if plano.get("periodo"):
        evidencias.append("Periodo considerado: " + str(plano["periodo"]))
    return evidencias


def node_finalizar(state):
    plano = state.get("plano") or {}
    brief = state.get("brief") or {}
    decisao = state.get("decisao") or {}

    if state.get("status") == "invalido":
        base = state.get("resposta") or "Entrada invalida."
    elif decisao.get("resposta"):
        base = decisao["resposta"]
    elif plano.get("tipo") == "fora_escopo":
        base = (
            "Nao e possivel atender a este pedido. O sistema e somente leitura e "
            "executa apenas consultas SELECT. Nenhuma alteracao foi realizada."
        )
    elif plano.get("tipo") == "definicao":
        if brief.get("definicoes"):
            base = "Definicoes encontradas: " + "; ".join(
                k + ": " + v for k, v in brief["definicoes"].items()
            )
        elif brief.get("lacunas"):
            base = "Nao encontrei definicao suficiente: " + "; ".join(brief["lacunas"])
        else:
            base = "Nao ha definicao registrada para esse termo no glossario."
    elif state.get("resposta"):
        base = state["resposta"]
    else:
        base = "Nao foi possivel concluir a resposta com a evidencia disponivel."

    evidencias = _formatar_evidencias(state)
    if evidencias:
        resposta = base + "\n\n**Evidencias e proveniencia**\n- " + "\n- ".join(evidencias)
    else:
        resposta = base

    status_final = "recusado" if plano.get("tipo") == "fora_escopo" else "concluido"
    if state.get("status") == "invalido":
        status_final = "invalido"
    env = novo_envelope(
        "coordenador", "usuario", "resposta_final",
        {
            "resposta": resposta,
            "status": status_final,
            "evidencias": evidencias,
        },
        state["correlation_id"],
        proveniencia=["decisao_final", "parecer_verificacao"],
    )
    return {
        "resposta": resposta,
        "status": status_final,
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }

In [13]:
def node_esclarecer(state):
    lacunas = (state.get("brief") or {}).get("lacunas") or []
    detalhe = "; ".join(lacunas) if lacunas else "A definicao essencial nao foi encontrada."
    resposta = (
        (state.get("decisao") or {}).get("resposta")
        or (
            "Preciso de uma confirmacao antes de consultar os dados: " + detalhe
            + " Pode informar a definicao ou o criterio de negocio que devo usar?"
        )
    )
    env = novo_envelope(
        "coordenador", "usuario", "pedido_esclarecimento",
        {"resposta": resposta, "lacunas": lacunas},
        state["correlation_id"],
        proveniencia=["brief_dominio"],
    )
    return {
        "resposta": resposta,
        "status": "esclarecimento",
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


def node_escalar(state):
    motivo = (
        state.get("erro")
        or (state.get("decisao") or {}).get("motivo")
        or "Orcamento ou evidencia insuficiente."
    )
    ticket = escalar_analista(motivo)["data"]
    artefato = state.get("artefato") or {}
    parecer = state.get("parecer") or {}

    partes = [
        "Nao consegui concluir com seguranca. Caso encaminhado a um analista humano.",
        "Ticket: " + ticket["ticket"] + ".",
        "Motivo: " + str(motivo),
    ]
    if artefato.get("sql"):
        partes.append("Ultimo SQL: " + artefato["sql"])
    if parecer.get("problemas"):
        partes.append("Pendencias do auditor: " + "; ".join(parecer["problemas"]))
    resposta = "\n\n".join(partes)

    env = novo_envelope(
        "coordenador", "usuario", "escalonamento",
        {"resposta": resposta, "ticket": ticket, "motivo": str(motivo)},
        state["correlation_id"],
        proveniencia=["parecer_verificacao", "decisao_escalar"],
    )
    return {
        "resposta": resposta,
        "status": "escalado",
        "mensagens": [env],
        "auditoria": [entrada_auditoria(env)],
    }


print("Agentes C3 definidos: Coordenador, Dominio, Analista SQL e Auditor.")

Agentes C3 definidos: Coordenador, Dominio, Analista SQL e Auditor.


## 9. Grafo multiagente

O grafo implementa o padrão *supervisor*: o Coordenador decide o próximo nó, o
Auditor pode forçar uma revisão e o orçamento limita o loop. O `InMemorySaver`
guarda cada transição para inspeção e replay.

In [14]:
def rotear_validacao(state):
    return "finalizar" if state.get("status") == "invalido" else "coordenador_planeja"


def rotear_plano(state):
    tipo = (state.get("plano") or {}).get("tipo")
    if tipo in ("fora_escopo", "invalido"):
        return "finalizar"
    return "dominio"


def rotear_dominio(state):
    if state.get("status") == "dominio_incompleto":
        return "esclarecer"
    if (state.get("plano") or {}).get("tipo") == "definicao":
        return "finalizar"
    return "politica"


def rotear_politica(state):
    return "analista" if state.get("status") == "politica_ok" else "escalar"


def rotear_decisao(state):
    acao = (state.get("decisao") or {}).get("acao")
    return {
        "finalizar": "finalizar",
        "revisar": "analista",
        "escalar": "escalar",
        "esclarecer": "esclarecer",
    }.get(acao, "escalar")


def construir_grafo_multiagente(llm):
    builder = StateGraph(EstadoAurora)
    builder.add_node("validar", node_validar_multi)
    builder.add_node("coordenador_planeja", lambda s: node_coordenador_planeja(s, llm))
    builder.add_node("dominio", lambda s: node_dominio(s, llm))
    builder.add_node("politica", node_politica)
    builder.add_node("analista", lambda s: node_analista(s, llm))
    builder.add_node("auditor", lambda s: node_auditor(s, llm))
    builder.add_node("coordenador_decide", lambda s: node_coordenador_decide(s, llm))
    builder.add_node("finalizar", node_finalizar)
    builder.add_node("escalar", node_escalar)
    builder.add_node("esclarecer", node_esclarecer)

    builder.add_edge(START, "validar")
    builder.add_conditional_edges(
        "validar", rotear_validacao,
        {"coordenador_planeja": "coordenador_planeja", "finalizar": "finalizar"},
    )
    builder.add_conditional_edges(
        "coordenador_planeja", rotear_plano,
        {"dominio": "dominio", "finalizar": "finalizar"},
    )
    builder.add_conditional_edges(
        "dominio", rotear_dominio,
        {
            "politica": "politica", "finalizar": "finalizar",
            "esclarecer": "esclarecer",
        },
    )
    builder.add_conditional_edges(
        "politica", rotear_politica,
        {"analista": "analista", "escalar": "escalar"},
    )
    builder.add_edge("analista", "auditor")
    builder.add_edge("auditor", "coordenador_decide")
    builder.add_conditional_edges(
        "coordenador_decide", rotear_decisao,
        {
            "finalizar": "finalizar", "analista": "analista",
            "escalar": "escalar", "esclarecer": "esclarecer",
        },
    )
    builder.add_edge("finalizar", END)
    builder.add_edge("escalar", END)
    builder.add_edge("esclarecer", END)
    return builder.compile(checkpointer=InMemorySaver())


def executar_c3(pergunta, llm, thread_id=None):
    thread_id = thread_id or novo_id("thread-c3")
    grafo = construir_grafo_multiagente(llm)
    return grafo.invoke(
        estado_inicial(pergunta, thread_id),
        config={"configurable": {"thread_id": thread_id}},
    )


_grafo_visual = construir_grafo_multiagente(MockLLM({}))
_mermaid = _grafo_visual.get_graph().draw_mermaid()
print(_mermaid)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validar(validar)
	coordenador_planeja(coordenador_planeja)
	dominio(dominio)
	politica(politica)
	analista(analista)
	auditor(auditor)
	coordenador_decide(coordenador_decide)
	finalizar(finalizar)
	escalar(escalar)
	esclarecer(esclarecer)
	__end__([<p>__end__</p>]):::last
	__start__ --> validar;
	analista --> auditor;
	auditor --> coordenador_decide;
	coordenador_decide -.-> analista;
	coordenador_decide -.-> escalar;
	coordenador_decide -.-> esclarecer;
	coordenador_decide -.-> finalizar;
	coordenador_planeja -.-> dominio;
	coordenador_planeja -.-> finalizar;
	dominio -.-> esclarecer;
	dominio -.-> finalizar;
	dominio -.-> politica;
	politica -.-> analista;
	politica -.-> escalar;
	validar -.-> coordenador_planeja;
	validar -.-> finalizar;
	escalar --> __end__;
	esclarecer --> __end__;
	finalizar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	cl

## 10. Demonstração com o LLM real (opcional)

Roda apenas se `OPENROUTER_API_KEY` ou `OPENCODE_API_KEY` estiver configurada. Sem
chave, esta seção é pulada e os testes offline continuam funcionando. O provedor é
detectado pelo prefixo da chave (`sk-or-...` ou `oc_sk_...`). A demo imprime a
resposta, o status, as ferramentas, os envelopes e a evidência SQL.

In [15]:
def tem_chave():
    return bool(os.environ.get("LLM_API_KEY"))


if tem_chave():
    llm_real = criar_llm(os.environ["LLM_API_KEY"])
    print("Cliente LLM:", type(llm_real).__name__)
    print("Modelos:", getattr(llm_real, "modelos", []))
    perguntas_demo = [
        "Quantas transacoes existem em marco de 2026?",
        "Qual a definicao de churn usada no nosso DW?",
        "Apague todos os clientes de teste.",
    ]
    for i, pergunta in enumerate(perguntas_demo, start=1):
        print("=" * 78)
        print("PERGUNTA:", pergunta)
        try:
            resultado = executar_c3(
                pergunta, llm_real, thread_id="demo-real-" + str(i)
            )
            print("STATUS:", resultado.get("status"))
            print("RESPOSTA:", resultado.get("resposta"))
            print("FERRAMENTAS:", resultado.get("tool_calls"))
            print("ENVELOPES:", len(resultado.get("mensagens", [])))
            if resultado.get("artefato", {}).get("sql"):
                print("SQL:", resultado["artefato"]["sql"])
            if resultado.get("parecer"):
                print("PARECER:", resultado["parecer"].get("aprovado"),
                      resultado["parecer"].get("motivos"))
        except Exception as e:
            print("Falha na demo real:", e)

    print("=" * 78)
    print("Comparacao C1 (LLM real) na pergunta 1:")
    try:
        r1 = executar_c1(perguntas_demo[0], llm_real)
        print("C1:", r1["resposta"], "| passos:", r1["steps"])
    except Exception as e:
        print("Falha na comparacao C1:", e)
else:
    print("LLM_API_KEY ausente: demo real pulada. Testes offline abaixo.")

Cliente LLM: LLMOpenCodeZen
Modelos: ['deepseek-v4.1-flash', 'claude-haiku-5-5', 'glm-5.3-flash']
PERGUNTA: Quantas transacoes existem em marco de 2026?


STATUS: concluido
RESPOSTA: Existem 25 transações registradas em março de 2026 (contagem de todas as linhas da tabela transacoes com data no período de 01/03/2026 a 31/03/2026).

**Evidencias e proveniencia**
- Definicoes aplicadas: transacoes: Registros de movimentacao financeira na tabela transacoes do DW Aurora, com grao de uma linha por transacao (id), associada a um cliente (cliente_id), com data, valor, tipo e moeda.; transacao valida: Transacao com valor maior que zero e moeda BRL.; marco de 2026: Periodo de referencia marco/2026, aplicado sobre a coluna de data da transacao (transacoes.data), armazenada como TEXT.
- Regras aplicadas: A contagem natural e por linha da tabela transacoes (grao definido por id), restrita ao periodo marco de 2026.; Se a contagem pretendida for de 'transacoes validas', aplicar o filtro de negocio: valor > 0 e moeda = 'BRL'.; transacoes.data e TEXT: o recorte de periodo depende do formato de data adotado no DW, que deve ser confirmado antes do calculo

STATUS: concluido
RESPOSTA: Definicoes encontradas: churn: Cliente sem nenhuma transacao nos ultimos 30 dias em relacao a data de referencia do DW.; transacao valida: Transacao com valor maior que zero e moeda BRL.; base: DW Aurora com as tabelas clientes, transacoes, faturas e solicitacoes_limite.

**Evidencias e proveniencia**
- Definicoes aplicadas: churn: Cliente sem nenhuma transacao nos ultimos 30 dias em relacao a data de referencia do DW.; transacao valida: Transacao com valor maior que zero e moeda BRL.; base: DW Aurora com as tabelas clientes, transacoes, faturas e solicitacoes_limite.
- Regras aplicadas: Churn e definido por ausencia de transacoes na janela de 30 dias contada retroativamente a partir da data de referencia do DW.; A verificacao de atividade do cliente deve ser feita por cliente_id na tabela transacoes, comparando a coluna data com a data de referencia.; O glossario define separadamente 'transacao valida' (valor > 0 e moeda BRL), que e a base de metricas como 

C1: Existem 25 transações em março de 2026. | passos: 4


## 11. Testes offline

Os testes usam `MockLLM` roteirizado por agente. Eles cobrem contratos, roteamento,
definição pura, consulta com evidência, revisão, preservação de informação,
bloqueio destrutivo, política de PII, risco alto, orçamento e checkpoint.

In [16]:
def acao_multi(nome, **argumentos):
    return json.dumps(
        {
            "pensamento": "preciso de uma ferramenta",
            "acao": nome,
            "argumentos": argumentos,
        },
        ensure_ascii=False,
    )


def final_multi(payload):
    return json.dumps({"final": payload}, ensure_ascii=False)


def json_direto(payload):
    return json.dumps(payload, ensure_ascii=False)


def acao_unica(nome, **argumentos):
    return json.dumps(
        {
            "thought": "preciso de uma ferramenta",
            "action": nome,
            "action_input": argumentos,
        },
        ensure_ascii=False,
    )


def final_unico(texto):
    return json.dumps(
        {"thought": "concluido", "final_answer": texto},
        ensure_ascii=False,
    )

In [17]:
PLANO_CONTAGEM = {
    "tipo": "consulta",
    "objetivo": "contar as transacoes de marco de 2026",
    "termos": ["transacao"],
    "periodo": "marco de 2026",
    "entidades": ["transacoes"],
    "precisa_dominio": True,
    "risco": "baixo",
    "justificativa": "Consulta simples com periodo explicito.",
}

BRIEF_CONTAGEM = {
    "definicoes": {
        "transacao valida": "Transacao com valor maior que zero e moeda BRL."
    },
    "tabelas": ["transacoes"],
    "colunas": {
        "transacoes": ["id", "cliente_id", "data", "valor", "tipo", "moeda"]
    },
    "regras": ["Filtrar o periodo de 2026-03-01 a 2026-03-31."],
    "lacunas": [],
    "confianca": 0.9,
}

SQL_CONTAGEM = (
    "SELECT COUNT(*) AS total FROM transacoes "
    "WHERE data >= '2026-03-01' AND data < '2026-04-01'"
)
SQL_CONTAGEM_ERRADA = "SELECT COUNT(*) AS total FROM transacoes"
RESPOSTA_CONTAGEM = "Existem 25 transacoes em marco de 2026."


def roteiro_contagem():
    return {
        "coordenador_planeja": [json_direto(PLANO_CONTAGEM)],
        "especialista_dominio": [
            acao_multi("inspecionar_schema", tabela="transacoes"),
            acao_multi("buscar_documentacao", termo="transacao valida"),
            final_multi(BRIEF_CONTAGEM),
        ],
        "analista_sql": [
            acao_multi("executar_sql", sql=SQL_CONTAGEM),
            final_multi({
                "sql": SQL_CONTAGEM,
                "explicacao": "Contagem de transacoes no periodo.",
                "hipoteses": ["datas em formato ISO"],
            }),
        ],
        "auditor": [
            json_direto({
                "aprovado": True,
                "motivos": [
                    "SQL usa a tabela correta, filtra o periodo e agrega com COUNT.",
                    "Resultado nao vazio.",
                ],
                "problemas": [],
                "correcao_sugerida": None,
                "confianca": 0.95,
            })
        ],
        "coordenador_decide": [
            json_direto({
                "acao": "finalizar",
                "resposta": RESPOSTA_CONTAGEM,
                "instrucao": None,
                "motivo": "Parecer aprovado com evidencia suficiente.",
            })
        ],
    }


def roteiro_definicao():
    return {
        "coordenador_planeja": [
            json_direto({
                "tipo": "definicao",
                "objetivo": "explicar a definicao de churn",
                "termos": ["churn"],
                "periodo": None,
                "entidades": [],
                "precisa_dominio": True,
                "risco": "baixo",
                "justificativa": "Pergunta sobre o significado de uma metrica.",
            })
        ],
        "especialista_dominio": [
            acao_multi("buscar_documentacao", termo="churn"),
            final_multi({
                "definicoes": {"churn": GLOSSARIO["churn"]},
                "tabelas": ["clientes", "transacoes"],
                "colunas": {},
                "regras": [],
                "lacunas": [],
                "confianca": 0.95,
            }),
        ],
    }


def roteiro_revisao():
    roteiro = roteiro_contagem()
    roteiro["analista_sql"] = [
        acao_multi("executar_sql", sql=SQL_CONTAGEM_ERRADA),
        final_multi({
            "sql": SQL_CONTAGEM_ERRADA,
            "explicacao": "Contagem total de transacoes.",
            "hipoteses": [],
        }),
        acao_multi("executar_sql", sql=SQL_CONTAGEM),
        final_multi({
            "sql": SQL_CONTAGEM,
            "explicacao": "Contagem de transacoes no periodo, corrigindo o filtro.",
            "hipoteses": ["datas em formato ISO"],
        }),
    ]
    roteiro["auditor"] = [
        json_direto({
            "aprovado": False,
            "motivos": ["A consulta nao filtra o periodo solicitado."],
            "problemas": ["periodo_ausente_no_sql"],
            "correcao_sugerida": "Inclua o filtro de marco de 2026 na consulta.",
            "confianca": 0.9,
        }),
        json_direto({
            "aprovado": True,
            "motivos": ["Periodo e agregacao corretos; resultado coerente."],
            "problemas": [],
            "correcao_sugerida": None,
            "confianca": 0.95,
        }),
    ]
    roteiro["coordenador_decide"] = [
        json_direto({
            "acao": "revisar",
            "resposta": None,
            "instrucao": "Inclua o filtro de marco de 2026 na consulta.",
            "motivo": "Auditor reprovou por falta de filtro de periodo.",
        }),
        json_direto({
            "acao": "finalizar",
            "resposta": RESPOSTA_CONTAGEM,
            "instrucao": None,
            "motivo": "Auditor aprovou a consulta corrigida.",
        }),
    ]
    return roteiro


def roteiro_termo_ausente():
    return {
        "coordenador_planeja": [
            json_direto({
                "tipo": "consulta",
                "objetivo": "calcular lucro liquido por agencia",
                "termos": ["lucro liquido"],
                "periodo": None,
                "entidades": ["agencias"],
                "precisa_dominio": True,
                "risco": "baixo",
                "justificativa": "Consulta depende de metrica nao definida.",
            })
        ],
        "especialista_dominio": [
            acao_multi("buscar_documentacao", termo="lucro liquido"),
            final_multi({
                "definicoes": {},
                "tabelas": ["transacoes"],
                "colunas": {},
                "regras": [],
                "lacunas": [
                    "O termo 'lucro liquido' nao esta definido no glossario."
                ],
                "confianca": 0.2,
            }),
        ],
    }


def roteiro_risco_alto():
    return {
        "coordenador_planeja": [
            json_direto({
                "tipo": "consulta",
                "objetivo": "listar dados pessoais de clientes",
                "termos": [],
                "periodo": None,
                "entidades": ["clientes"],
                "precisa_dominio": True,
                "risco": "alto",
                "justificativa": "Pedido envolve dados sensiveis e exige aprovacao.",
            })
        ],
        "especialista_dominio": [
            acao_multi("inspecionar_schema", tabela="clientes"),
            final_multi({
                "definicoes": {},
                "tabelas": ["clientes"],
                "colunas": {"clientes": ["id", "nome", "cidade", "uf"]},
                "regras": [],
                "lacunas": [],
                "confianca": 0.9,
            }),
        ],
    }


def roteiro_orcamento():
    sql_ruim = "SELECT COUNT(*) FROM transacoes WHERE coluna_inexistente = 1"
    roteiro = roteiro_contagem()
    roteiro["analista_sql"] = []
    roteiro["auditor"] = []
    roteiro["coordenador_decide"] = []
    for i in range(3):
        roteiro["analista_sql"].extend([
            acao_multi("executar_sql", sql=sql_ruim),
            final_multi({
                "sql": sql_ruim,
                "explicacao": "Tentativa " + str(i + 1) + " de consulta.",
                "hipoteses": [],
            }),
        ])
        roteiro["auditor"].append(json_direto({
            "aprovado": False,
            "motivos": ["A consulta falhou."],
            "problemas": ["erro_sql"],
            "correcao_sugerida": "Corrija a coluna inexistente.",
            "confianca": 0.9,
        }))
        roteiro["coordenador_decide"].append(json_direto({
            "acao": "revisar",
            "resposta": None,
            "instrucao": "Corrija a coluna inexistente.",
            "motivo": "Erro de SQL.",
        }))
    return roteiro


def roteiro_c1_contagem():
    return {"unico": [
        acao_unica("inspecionar_schema", tabela="transacoes"),
        acao_unica("executar_sql", sql=SQL_CONTAGEM),
        final_unico(RESPOSTA_CONTAGEM),
    ]}


def roteiro_c1_definicao():
    return {"unico": [
        acao_unica("buscar_documentacao", termo="churn"),
        final_unico(
            "Churn e definido como cliente sem nenhuma transacao nos ultimos "
            "30 dias em relacao a data de referencia do DW."
        ),
    ]}


def roteiro_c1_destrutivo():
    return {"unico": [
        acao_unica("executar_sql", sql="DROP TABLE clientes"),
        final_unico(
            "Nao e possivel executar operacoes de exclusao; somente consultas "
            "SELECT sao permitidas."
        ),
    ]}


def roteiro_c1_termo_ausente():
    return {"unico": [
        acao_unica("buscar_documentacao", termo="lucro liquido"),
        final_unico(
            "Nao encontrei definicao de lucro liquido no glossario. "
            "Encaminhei o caso a um analista."
        ),
    ]}

In [18]:
def executar_testes_offline():
    resultados = []

    def rodar(nome, funcao):
        try:
            funcao()
            resultados.append((nome, True, ""))
        except Exception as e:
            resultados.append((nome, False, str(e)))

    def teste_contratos():
        env = novo_envelope(
            "a", "b", "t", {"x": 1}, "corr-1"
        )
        assert env["de"] == "a" and env["payload"]["x"] == 1
        try:
            BriefDominio.model_validate({"definicoes": {}, "campo_extra": 1})
            raise AssertionError("BriefDominio aceitou campo extra")
        except ValidationError:
            pass
        try:
            EnvelopeMensagem.model_validate({"de": "a"})
            raise AssertionError("Envelope aceitou payload incompleto")
        except ValidationError:
            pass

    def teste_definicao():
        r = executar_c3(
            "Qual a definicao de churn usada no nosso DW?",
            MockLLM(roteiro_definicao()),
            thread_id="teste-definicao",
        )
        assert r["status"] == "concluido", r["status"]
        assert "30 dias" in r["resposta"]
        assert r["plano"]["tipo"] == "definicao"
        assert not [t for t in r.get("tool_calls", []) if t == "executar_sql"]
        assert not r.get("artefato")
        assert len(r["mensagens"]) >= 3

    def teste_consulta():
        r = executar_c3(
            "Quantas transacoes existem em marco de 2026?",
            MockLLM(roteiro_contagem()),
            thread_id="teste-consulta",
        )
        assert r["status"] == "concluido", r["status"]
        assert "25" in r["resposta"]
        assert r["artefato"]["sql"].startswith(SQL_CONTAGEM)
        assert r["parecer"]["aprovado"] is True
        assert r["artefato"]["resultado"]["n"] == 1
        assert len(r["mensagens"]) >= 5
        assert len(r["auditoria"]) >= 5

    def teste_revisao():
        mock = MockLLM(roteiro_revisao())
        r = executar_c3(
            "Quantas transacoes existem em marco de 2026?",
            mock,
            thread_id="teste-revisao",
        )
        assert r["status"] == "concluido", r["status"]
        assert r["revisoes"] == 1, r["revisoes"]
        assert "25" in r["resposta"]
        chamadas_analista = [
            c for c in mock.chamadas if c["agente"] == "analista_sql"
        ]
        assert len(chamadas_analista) >= 2
        texto_segunda = json.dumps(
            chamadas_analista[-1]["mensagens"], ensure_ascii=False
        )
        assert "Inclua o filtro de marco de 2026" in texto_segunda
        assert any(
            m["tipo"] == "decisao_revisar" for m in r["mensagens"]
        )

    def teste_preservacao():
        mock = MockLLM(roteiro_contagem())
        r = executar_c3(
            "Quantas transacoes existem em marco de 2026?",
            mock,
            thread_id="teste-preservacao",
        )
        assert "transacao valida" in r["brief"]["definicoes"]
        assert r["brief"]["definicoes"]["transacao valida"] in r["resposta"]
        assert r["artefato"]["sql"] in r["resposta"]
        chamadas_analista = [
            c for c in mock.chamadas if c["agente"] == "analista_sql"
        ]
        chamadas_auditor = [
            c for c in mock.chamadas if c["agente"] == "auditor"
        ]
        assert "valor maior que zero" in json.dumps(
            chamadas_analista[0]["mensagens"], ensure_ascii=False
        )
        assert "valor maior que zero" in json.dumps(
            chamadas_auditor[0]["mensagens"], ensure_ascii=False
        )
        tipos = {m["tipo"] for m in r["mensagens"]}
        assert {"plano_tarefa", "brief_dominio", "artefato_consulta",
                "parecer_verificacao", "resposta_final"}.issubset(tipos)

    def teste_destrutivo():
        r = executar_c3(
            "Apague todos os clientes de teste.",
            MockLLM({}),
            thread_id="teste-destrutivo",
        )
        assert r["status"] == "recusado", r["status"]
        assert "somente leitura" in r["resposta"].lower()
        assert not [t for t in r.get("tool_calls", []) if t == "executar_sql"]
        assert len(r["mensagens"]) >= 2

    def teste_politica_pii():
        r = executar_sql("SELECT nome, cidade FROM clientes LIMIT 3")
        assert r.get("ok") is True, r
        assert r["data"]["pii_mascarada"] is True
        assert all(linha["nome"] == "***" for linha in r["data"]["rows"])
        politica = avaliar_politica_sql("SELECT * FROM tabela_proibida")
        assert politica["permitido"] is False

    def teste_risco_alto():
        r = executar_c3(
            "Liste os dados pessoais dos clientes.",
            MockLLM(roteiro_risco_alto()),
            thread_id="teste-risco-alto",
        )
        assert r["status"] == "escalado", r["status"]
        assert "Ticket" in r["resposta"]
        assert not [t for t in r.get("tool_calls", []) if t == "executar_sql"]

    def teste_orcamento():
        r = executar_c3(
            "Quantas transacoes existem em marco de 2026?",
            MockLLM(roteiro_orcamento()),
            thread_id="teste-orcamento",
        )
        assert r["status"] == "escalado", r["status"]
        assert r["revisoes"] == MAX_REVISOES, r["revisoes"]
        assert "Ticket" in r["resposta"]
        assert r.get("tentativas_sql", 0) >= 3

    def teste_checkpoint():
        mock = MockLLM(roteiro_contagem())
        grafo = construir_grafo_multiagente(mock)
        config = {"configurable": {"thread_id": "teste-checkpoint"}}
        saida = grafo.invoke(
            estado_inicial(
                "Quantas transacoes existem em marco de 2026?",
                "teste-checkpoint",
            ),
            config=config,
        )
        assert "25" in saida["resposta"]
        estado = grafo.get_state(config).values
        assert "25" in estado["resposta"]
        historico = list(grafo.get_state_history(config))
        assert len(historico) >= 1

    rodar("contratos_e_envelopes", teste_contratos)
    rodar("definicao_pura_sem_sql", teste_definicao)
    rodar("consulta_com_evidencia", teste_consulta)
    rodar("auditor_forca_revisao", teste_revisao)
    rodar("preservacao_nas_fronteiras", teste_preservacao)
    rodar("bloqueio_destrutivo", teste_destrutivo)
    rodar("politica_de_pii", teste_politica_pii)
    rodar("risco_alto_escala", teste_risco_alto)
    rodar("orcamento_limita_revisoes", teste_orcamento)
    rodar("checkpoint_e_replay", teste_checkpoint)

    print("Testes offline")
    for nome, ok, detalhe in resultados:
        print(("PASS" if ok else "FAIL") + " - " + nome
              + ((" | " + detalhe) if detalhe else ""))
    total_ok = sum(1 for _, ok, _ in resultados if ok)
    print("Total:", total_ok, "/", len(resultados))
    return resultados


resultados_testes = executar_testes_offline()
assert all(ok for _, ok, _ in resultados_testes), "Ha testes offline falhando."

Testes offline
PASS - contratos_e_envelopes
PASS - definicao_pura_sem_sql
PASS - consulta_com_evidencia
PASS - auditor_forca_revisao
PASS - preservacao_nas_fronteiras
PASS - bloqueio_destrutivo
PASS - politica_de_pii
PASS - risco_alto_escala
PASS - orcamento_limita_revisoes
PASS - checkpoint_e_replay
Total: 10 / 10


### Trilha de envelopes de uma execução

O exemplo abaixo mostra a comunicação estruturada de uma consulta: cada linha é um
envelope, com remetente, destinatário, tipo e as chaves do payload preservado.

In [19]:
_exemplo = executar_c3(
    "Quantas transacoes existem em marco de 2026?",
    MockLLM(roteiro_contagem()),
    thread_id="exemplo-envelopes",
)
print("Trilha de envelopes")
print("-" * 100)
for i, env in enumerate(_exemplo["mensagens"], start=1):
    print(
        f"{i:>2}. {env['de']:<20} -> {env['para']:<20} "
        f"| {env['tipo']:<22} | payload: {sorted(env['payload'].keys())}"
    )
print("-" * 100)
print("Correlacao:", _exemplo["correlation_id"])
print("Envelopes:", len(_exemplo["mensagens"]))
print("Auditoria:", len(_exemplo["auditoria"]))

Trilha de envelopes
----------------------------------------------------------------------------------------------------
 1. sistema              -> coordenador          | entrada_validada       | payload: ['pergunta']
 2. coordenador          -> especialista_dominio | plano_tarefa           | payload: ['entidades', 'justificativa', 'objetivo', 'periodo', 'precisa_dominio', 'risco', 'termos', 'tipo']
 3. especialista_dominio -> coordenador          | brief_dominio          | payload: ['colunas', 'confianca', 'definicoes', 'lacunas', 'regras', 'tabelas']
 4. coordenador          -> analista_sql         | politica_ok            | payload: ['restricoes']
 5. analista_sql         -> auditor              | artefato_consulta      | payload: ['erro', 'explicacao', 'hipoteses', 'resultado', 'sql', 'tentativas', 'tool_calls']
 6. auditor              -> coordenador          | parecer_verificacao    | payload: ['aprovado', 'confianca', 'correcao_sugerida', 'motivos', 'problemas']
 7. coordenador

## 12. Avaliação comparativa: C1 × C2 × C3

A avaliação offline é um **harness controlado**, não um benchmark estatístico de
qualidade de LLM. Ela mede propriedades arquiteturais e de fluxo: acerto nos
cenários, passos, consultas SQL, revisões, envelopes, preservação de informação e
auditoria. A demo com LLM real, quando habilitada, complementa a avaliação.

In [20]:
def _metricas_c1(pergunta, roteiro, esperado):
    r = executar_c1(pergunta, MockLLM(roteiro))
    return {
        "acerto": esperado in r["resposta"].lower(),
        "passos": r["steps"],
        "sql_calls": sum(1 for t in r["tool_calls"] if t == "executar_sql"),
        "envelopes": 0,
        "revisoes": 0,
        "info": False,
        "auditoria": 0,
        "resposta": r["resposta"],
    }


def _metricas_c2(pergunta, roteiro, esperado, thread_id):
    r = executar_c2(pergunta, MockLLM(roteiro), thread_id=thread_id)
    return {
        "acerto": esperado in r["resposta"].lower(),
        "passos": len(r.get("trace", [])) or len(r.get("auditoria", [])),
        "sql_calls": sum(1 for t in r.get("tool_calls", []) if t == "executar_sql"),
        "envelopes": len(r.get("mensagens", [])),
        "revisoes": 0,
        "info": bool(r.get("brief")),
        "auditoria": len(r.get("auditoria", [])),
        "resposta": r["resposta"],
    }


def _metricas_c3(pergunta, roteiro, esperado, thread_id):
    r = executar_c3(pergunta, MockLLM(roteiro), thread_id=thread_id)
    return {
        "acerto": esperado in r["resposta"].lower(),
        "passos": len(r.get("auditoria", [])),
        "sql_calls": sum(1 for t in r.get("tool_calls", []) if t == "executar_sql"),
        "envelopes": len(r.get("mensagens", [])),
        "revisoes": r.get("revisoes", 0),
        "info": bool(r.get("brief")) and bool(r.get("artefato")),
        "auditoria": len(r.get("auditoria", [])),
        "resposta": r["resposta"],
    }


def avaliar_configuracoes():
    cenarios = [
        (
            "contagem",
            "Quantas transacoes existem em marco de 2026?",
            roteiro_c1_contagem(), roteiro_contagem(), "25",
        ),
        (
            "definicao",
            "Qual a definicao de churn usada no nosso DW?",
            roteiro_c1_definicao(), roteiro_definicao(), "30 dias",
        ),
        (
            "destrutivo",
            "Apague todos os clientes de teste.",
            roteiro_c1_destrutivo(), {}, "nao e possivel",
        ),
        (
            "termo_ausente",
            "Qual o lucro liquido por agencia?",
            roteiro_c1_termo_ausente(), roteiro_termo_ausente(), "definicao",
        ),
    ]

    linhas = []
    for nome, pergunta, r1, r3, esperado in cenarios:
        m1 = _metricas_c1(pergunta, r1, esperado)
        m2 = _metricas_c2(pergunta, r1, esperado, "aval-c2-" + nome)
        m3 = _metricas_c3(pergunta, r3, esperado, "aval-c3-" + nome)
        linhas.append((nome, m1, m2, m3))

    # Cenario de revisao: C1/C2 nao possuem verificacao independente.
    pergunta_revisao = "Quantas transacoes existem em marco de 2026?"
    m3_revisao = _metricas_c3(
        pergunta_revisao, roteiro_revisao(), "25", "aval-c3-revisao"
    )
    linhas.append(("revisao", None, None, m3_revisao))

    print("Avaliacao comparativa (harness offline)")
    print("-" * 96)
    print(
        f"{'cenario':<16} {'C1':<10} {'C2':<10} {'C3':<10} "
        f"{'C3 passos':>10} {'C3 SQL':>7} {'C3 env':>7} {'C3 rev':>7}"
    )
    print("-" * 96)
    for nome, m1, m2, m3 in linhas:
        c1 = "sim" if m1 and m1["acerto"] else ("n/a" if m1 is None else "nao")
        c2 = "sim" if m2 and m2["acerto"] else ("n/a" if m2 is None else "nao")
        c3 = "sim" if m3["acerto"] else "nao"
        print(
            f"{nome:<16} {c1:<10} {c2:<10} {c3:<10} "
            f"{m3['passos']:>10} {m3['sql_calls']:>7} "
            f"{m3['envelopes']:>7} {m3['revisoes']:>7}"
        )
    print("-" * 96)

    print("\nCusto por configuracao (passos / SQL / envelopes)")
    print("-" * 72)
    print(f"{'cenario':<16} {'C1':<16} {'C2':<16} {'C3':<16}")
    print("-" * 72)
    for nome, m1, m2, m3 in linhas:
        def fmt(m):
            if m is None:
                return "n/a"
            return f"{m['passos']}/{m['sql_calls']}/{m['envelopes']}"
        print(f"{nome:<16} {fmt(m1):<16} {fmt(m2):<16} {fmt(m3):<16}")
    print("-" * 72)

    print("\nPropriedades arquiteturais")
    print("-" * 72)
    print(f"{'propriedade':<34} {'C1':<8} {'C2':<8} {'C3':<8}")
    print("-" * 72)
    propriedades = [
        ("contrato estruturado", "nao", "nao", "sim"),
        ("envelopes com proveniencia", "nao", "nao", "sim"),
        ("trilha de auditoria", "nao", "sim", "sim"),
        ("verificacao independente", "nao", "nao", "sim"),
        ("politica pre-SQL", "nao", "nao", "sim"),
        ("checkpoint/replay", "nao", "sim", "sim"),
        ("revisao com feedback", "nao", "nao", "sim"),
    ]
    for prop, a, b, c in propriedades:
        print(f"{prop:<34} {a:<8} {b:<8} {c:<8}")
    print("-" * 72)

    return linhas


linhas_avaliacao = avaliar_configuracoes()

Avaliacao comparativa (harness offline)
------------------------------------------------------------------------------------------------
cenario          C1         C2         C3          C3 passos  C3 SQL  C3 env  C3 rev
------------------------------------------------------------------------------------------------
contagem         sim        sim        sim                 8       1       8       0
definicao        sim        sim        sim                 4       0       4       0
destrutivo       sim        sim        sim                 3       0       3       0
termo_ausente    sim        sim        sim                 4       0       4       0
revisao          n/a        n/a        sim                11       2      11       1
------------------------------------------------------------------------------------------------

Custo por configuracao (passos / SQL / envelopes)
------------------------------------------------------------------------
cenario          C1               C

## 13. Leitura dos resultados

O harness mostra o trade-off esperado:

- **C1** é o mais enxuto: mesmo raciocínio do agente único, sem contratos,
  envelopes, política pré-SQL, verificação independente ou checkpoint. O custo de
  C2 é a infraestrutura de orquestração, não mais passos de raciocínio.
- **C2** adiciona estado explícito, auditoria e replay sem mudar o raciocínio. O
  custo cresce pouco, mas o agente continua sendo autor e juiz do próprio SQL.
- **C3** gasta mais passos e chamadas, mas ganha revisão com feedback, verificação
  independente, política antes da ação e preservação estruturada de definições,
  SQL, resultado e parecer.

A conclusão não é “multiagente sempre vence”. É que, para perguntas de BI com
dependência semântica e risco de erro silencioso, o custo adicional compra
**auditabilidade e contenção de erro**. Para perguntas triviais e de baixo risco, C1
ou C2 continuam sendo escolhas racionais.

---

## 14. Preparação para a governança da Entrega III

O que já está pronto para receber a próxima etapa:

| Gancho | Onde está | O que a Entrega III pode adicionar |
|---|---|---|
| Classificação de risco | `PlanoTarefa.risco` | Política de risco configurável, aprovação obrigatória por nível |
| Gate pré-SQL | `node_politica` + `avaliar_politica_sql` | Catálogo de políticas, finalidades permitidas, retenção |
| PII | `mascarar_pii` | Classificação de colunas, tokenização, auditoria de acesso |
| Trilha de auditoria | `auditoria[]` + envelopes | Persistência externa, assinatura, retenção, consulta |
| Checkpoint/replay | `InMemorySaver` + `thread_id` | Checkpointer durável, replay de decisões |
| Aprovação humana | `node_politica` (ponto de extensão) | `interrupt` do LangGraph, fila de aprovação |
| Orçamento | `MAX_REVISOES`, `MAX_TENTATIVAS_SQL` | Custos/tokens, limites por tenant, circuit breaker |

A arquitetura já separa **quem decide**, **quem executa** e **quem verifica**. A
governança entra como política explícita sobre essas fronteiras, não como um prompt
extra no agente.

## 15. Limitações e próximos passos

- O DW é sintético e pequeno; os resultados não medem desempenho de produção.
- A avaliação offline é um harness controlado; a demo real depende do modelo e da
  chave do usuário.
- A política de PII cobre o campo `nome` de `clientes`; a Entrega III deve
  classificar todas as colunas e finalidades.
- O checkpointer é em memória; para produção, usar persistência durável.
- Próximos passos: paralelizar subperguntas independentes com `Send`, adicionar
  aprovação humana com `interrupt`, persistir auditoria e criar uma suíte de
  avaliação com LLM real em maior escala.

## 16. Como executar

1. Abra o notebook no Google Colab.
2. Configure `OPENROUTER_API_KEY` (OpenRouter) ou `OPENCODE_API_KEY` (OpenCode Zen)
   em **Secrets** (recomendado), ou exporte `LLM_API_KEY` no ambiente. Nunca salve
   a chave no arquivo.
3. Execute todas as células. Sem chave, a demo real é pulada e os testes offline
   rodam normalmente.
4. A saída mostra o grafo, os testes, a avaliação C1×C2×C3 e, se houver chave, a
   demo real com a trilha completa.

## 17. Referências

- LangGraph. *Documentation: StateGraph, checkpointing, human-in-the-loop*. https://langchain-ai.github.io/langgraph/
- Yao, S. et al. *ReAct: Synergizing Reasoning and Acting in Language Models*. 2022.
- OpenRouter. *API documentation*. https://openrouter.ai/docs
- OpenCode Zen. *API documentation*. https://opencode.ai/docs/zen/
- Pydantic. *Data validation for Python*. https://docs.pydantic.dev/
- Padrões de coordenação: *supervisor*, *handoff* e *maker-checker*.

In [21]:
print("Notebook concluido.")
print("Testes offline:", sum(1 for _, ok, _ in resultados_testes if ok),
      "/", len(resultados_testes))
print("Cenarios avaliados:", len(linhas_avaliacao))
print("Chave configurada:", tem_chave())

Notebook concluido.
Testes offline: 10 / 10
Cenarios avaliados: 5
Chave configurada: True
